In [ ]:
# synthetic_offset_srvf_pipeline.py
"""
Repeated Synthetic Offset-SRVF Pipeline
======================================

GitHub-ready simulation script for the embedded Offset-SRVF framework.

Pipeline
--------
1. Generate synthetic closed inner curves beta_i(s).
2. Generate scalar thickness functions r_i(s).
3. Construct outer curves using normal offset.
4. Embed each curve-thickness pair:

       X_i(s) = ( beta_i(s), alpha r_i(s) ) in R^3.

5. Compute SRVF representations.
6. Align embedded SRVFs using penalized dynamic-programming warping.
7. Apply SO(2) rotation to geometric coordinates.
8. Select lambda by K-fold CV using Hilbert-sphere SRVF geodesic error.
9. Compute Karcher mean of aligned SRVFs on the Hilbert sphere.
10. Align full data using selected lambda.
11. Save figures, tables, LaTeX snippets, and repeated summaries.

Author
------
Bahadir Yuzbasi
"""

# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import json
import numpy as np
import sys
NUMERICS_DIR = Path("methods").resolve()
if str(NUMERICS_DIR) not in sys.path:
    sys.path.insert(0, str(NUMERICS_DIR))
import srvf_numerics as _srvf_numerics
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

from sklearn.model_selection import KFold


try:
    import fdasrsf as fs
    HAS_FDASRSF = True
except Exception:
    HAS_FDASRSF = False


# ============================================================
# 1. Global configuration
# ============================================================

SEED = 123
rng = np.random.default_rng(SEED)

K = 50
N = 250

# For testing: N_SIM = 1 or 2
# For final paper: N_SIM = 20, 30, or 50
N_SIM = 10
N_FOLDS = 5

s = np.linspace(0.0, 1.0, N)
dt = 1.0 / (N - 1)

alpha = 4.0

PHASE_MODE = "moderate"
AMPLITUDE_MODE = "moderate"

LAMBDA_GRID = [
    0.0,
    0.002, 0.005,
    0.01, 0.02, 0.05,
    0.1, 0.2, 0.5,
    1.0, 2.0, 4.0,
    8.0, 16.0
]
#LAMBDA_GRID = [0.5]

N_ITER_ALIGN = 4
USE_CYCLIC_PREALIGNMENT = True
RANDOM_STATE = 2026

ADD_NOISE = False
NOISE_LEVEL = 0.005

OUT_DIR = Path("runs/synthetic")
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
DATA_DIR = OUT_DIR / "data"
TEX_DIR = OUT_DIR / "tex"

# Each complete run requires a new, empty destination.
OUT_DIR.mkdir(parents=True, exist_ok=False)
for d in [FIG_DIR, TAB_DIR, DATA_DIR, TEX_DIR]:
    d.mkdir(parents=False, exist_ok=False)

mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["pdf.fonttype"] = 42


# ============================================================
# 2. Saving utilities
# ============================================================

def save_figure(fig, name, caption=None, label=None):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.eps", format="eps", bbox_inches="tight")

    if caption is not None and label is not None:
        tex = rf"""
\begin{{figure}}[!ht]
\centering
\includegraphics[width=0.95\textwidth]{{figures/{name}.pdf}}
\caption{{{caption}}}
\label{{{label}}}
\end{{figure}}
"""
        with open(TEX_DIR / f"{name}.tex", "w") as f:
            f.write(tex.strip() + "\n")


def save_latex_table(df, name, caption, label):
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)

    tex = df.to_latex(
        index=False,
        float_format="%.6f",
        caption=caption,
        label=label
    )

    with open(TEX_DIR / f"{name}.tex", "w") as f:
        f.write(tex)


def save_json(obj, name):
    with open(DATA_DIR / f"{name}.json", "w") as f:
        json.dump(obj, f, indent=4)


# ============================================================
# 3. Basic geometry
# ============================================================

def close_for_plot(Y):
    if np.allclose(Y[0], Y[-1]):
        return Y
    return np.vstack([Y, Y[0]])


def finite_difference_periodic(Y):
    """
    Periodic central difference for closed curves on [0,1]
    with duplicated endpoint Y[-1] = Y[0].
    """
    Y = np.asarray(Y)
    dY = np.zeros_like(Y)

    # Use only unique periodic samples
    Yu = Y[:-1]
    M = len(Yu)
    h = 1.0 / M

    dYu = (np.roll(Yu, -1, axis=0) - np.roll(Yu, 1, axis=0)) / (2.0 * h)

    dY[:-1] = dYu
    dY[-1] = dY[0]

    return dY


def signed_area(beta):
    x = beta[:, 0]
    y = beta[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - y * np.roll(x, -1))


def ensure_ccw(beta):
    if signed_area(beta) < 0:
        return beta[::-1].copy()
    return beta


def unit_tangent_normal(beta):
    db = finite_difference_periodic(beta)

    speed = np.linalg.norm(db, axis=1, keepdims=True)
    speed = np.maximum(speed, 1e-12)

    T = db / speed
    Nvec = np.column_stack([T[:, 1], -T[:, 0]])

    centroid = beta.mean(axis=0, keepdims=True)
    radial = beta - centroid

    if np.mean(np.sum(Nvec * radial, axis=1)) < 0:
        Nvec = -Nvec

    return T, Nvec


def reconstruct_outer(beta, r):
    _, Nvec = unit_tangent_normal(beta)
    beta_out = beta + r[:, None] * Nvec
    beta_out[-1] = beta_out[0]
    return beta_out


def embed_curve(beta, r):
    """
    Embedded closed curve:
        X(s) = (beta(s), alpha r(s)).

    Closure requires:
        beta(1)=beta(0),
        r(1)=r(0),
        therefore X(1)=X(0).
    """
    beta = enforce_curve_closure(beta)
    r = enforce_scalar_periodicity(r)

    X = np.column_stack([beta, alpha * r])
    X[-1] = X[0]

    return X


def circular_smooth(x, n_iter=2):
    """Smooth unique periodic nodes, then copy the closing value."""
    y = x[:-1].copy()
    for _ in range(n_iter):
        y = 0.25 * np.roll(y, 1) + 0.50 * y + 0.25 * np.roll(y, -1)
    return np.concatenate([y, y[:1]])


def compute_curvature_periodic(beta):
    """Estimate curvature on the unique nodes of an endpoint-inclusive grid."""
    unique = beta[:-1]
    h = 1.0 / len(unique)
    d1 = finite_difference_periodic(beta)[:-1]
    d2 = (
        np.roll(unique, -1, axis=0)
        - 2.0 * unique
        + np.roll(unique, 1, axis=0)
    ) / h**2

    x1, y1 = d1[:, 0], d1[:, 1]
    x2, y2 = d2[:, 0], d2[:, 1]
    num = x1 * y2 - y1 * x2
    den = (x1**2 + y1**2)**1.5 + 1e-10
    curvature = num / den
    return np.concatenate([curvature, curvature[:1]])


def fdasrsf_refine_alignment(X_aligned):
    """
    Refine alignment using fdasrsf elastic curve framework.

    Input:
        X_aligned: array with shape (K, N, 3)

    Returns:
        X_refined: array with shape (K, N, 3)
        gammas_refined: array with shape (K, N), if available
        mean_curve: array with shape (N, 3)
    """

    if not HAS_FDASRSF:
        print("fdasrsf not available -> skipping refinement")
        return X_aligned, None, None

    try:
        # Ensure closure before sending to fdasrsf
        X_closed = np.array([
            enforce_curve_closure(X_aligned[i])
            for i in range(len(X_aligned))
        ])

        # fdasrsf expects shape (dim, N, K)
        X_fdasrsf = np.transpose(X_closed, (2, 1, 0))

        # Old code used fs.fdacurve(..., mode='O', N=N, scale=False)
        obj = fs.fdacurve(X_fdasrsf, mode="O", N=N, scale=False)

        # Elastic alignment / Karcher mean
        obj.srvf_align(rotation=False)

        # Aligned curves
        if hasattr(obj, "beta"):
            X_refined = np.transpose(obj.beta, (2, 1, 0))
        else:
            X_refined = X_closed

        X_refined = np.array([
            enforce_curve_closure(X_refined[i])
            for i in range(len(X_refined))
        ])

        # Warping functions: fdasrsf versions may use gams, not gam
        gammas_refined = None

        if hasattr(obj, "gams"):
            gammas_refined = obj.gams

        elif hasattr(obj, "gam"):
            gammas_refined = obj.gam

        if gammas_refined is not None:
            gammas_refined = np.asarray(gammas_refined)

            # Common fdasrsf output is (N, K), convert to (K, N)
            if gammas_refined.shape == (N, len(X_aligned)):
                gammas_refined = gammas_refined.T

            # If already (K, N), leave as is
            elif gammas_refined.shape == (len(X_aligned), N):
                pass

            else:
                print(
                    "Warning: fdasrsf gamma shape not recognized:",
                    gammas_refined.shape
                )
                gammas_refined = None

        # Mean curve
        mean_curve = None

        if hasattr(obj, "mu"):
            mu = np.asarray(obj.mu)

            if mu.shape == (3, N):
                mean_curve = mu.T
            elif mu.shape == (N, 3):
                mean_curve = mu
            else:
                mean_curve = np.mean(X_refined, axis=0)
        else:
            mean_curve = np.mean(X_refined, axis=0)

        mean_curve = enforce_curve_closure(mean_curve)

        return X_refined, gammas_refined, mean_curve

    except Exception as e:
        print("fdasrsf refinement failed. Using DP alignment.")
        print("Reason:", str(e))
        return X_aligned, None, None

def endpoint_closure_error(curve):
    """
    Endpoint closure error ||X(1)-X(0)||.
    """
    return np.linalg.norm(curve[-1] - curve[0])


def enforce_curve_closure(curve):
    """
    Enforce endpoint closure by subtracting a linear correction.
    """
    t = np.linspace(0.0, 1.0, len(curve))[:, None]
    e = curve[-1] - curve[0]
    curve_closed = curve - t * e
    curve_closed[-1] = curve_closed[0]
    return curve_closed


def enforce_scalar_periodicity(r):
    """
    Enforce r(1)=r(0) for scalar thickness function.
    """
    r = r.copy()
    r[-1] = r[0]
    return r


def closure_vector_from_srvf(q):
    """
    Closed-curve SRVF condition:
        integral q(s)||q(s)|| ds = 0.

    Use trapezoidal integration because the grid includes both endpoints.
    """
    vel = q * np.linalg.norm(q, axis=1, keepdims=True)
    return np.trapezoid(vel, s, axis=0)

def srvf_closure_error_from_curve(X):
    """
    Numerically stable closed-curve diagnostic for an embedded curve.

    For a valid embedded closed curve, the primary closure condition is
        X(1) = X(0).

    This avoids artificial nonzero closure caused by finite-difference
    SRVF reconstruction on a grid with duplicated endpoints.
    """
    return endpoint_closure_error(X)


def warping_penalty_integral(gamma):
    """
    Integral penalty:
        P(gamma) = integral_0^1 (gamma_dot(t) - 1)^2 dt

    Discrete approximation.
    """
    gamma_dot = np.gradient(gamma, s)
    return np.trapezoid((gamma_dot - 1.0) ** 2, s)


def compute_final_penalized_karcher_mean(X_aligned, gammas, lambda_star):
    """
    Final penalized Karcher mean for the selected optimal lambda.

    The lambda is used through the aligned representatives:
        q_i^{lambda} = q_i * gamma_i^{lambda}

    Then:
        mu_lambda = argmin_mu sum_i d_S^2(mu, q_i^{lambda})
    """

    q_aligned = np.array([
        normalize_srvf(srvf(X_aligned[i]))
        for i in range(len(X_aligned))
    ])

    q_mean_lambda = karcher_mean_sphere(
        q_aligned,
        max_iter=100,
        tol=1e-6,
        verbose=True
    )

    distances = np.array([
        sphere_geodesic_distance(q_aligned[i], q_mean_lambda)
        for i in range(len(q_aligned))
    ])

    penalties = np.array([
        warping_penalty_integral(gammas[i])
        for i in range(len(gammas))
    ])

    summary = {
        "lambda_star": lambda_star,
        "mean_srvf_geodesic_distance": distances.mean(),
        "sd_srvf_geodesic_distance": distances.std(),
        "mean_warping_penalty": penalties.mean(),
        "sd_warping_penalty": penalties.std()
    }

    return q_mean_lambda, q_aligned, distances, penalties, summary

def compute_final_penalized_karcher_mean_from_q(q_aligned, gammas, lambda_star):
    """
    Final Karcher mean computed directly from aligned SRVFs q_i^a.
    """

    q_aligned = np.array([
        normalize_srvf(q_aligned[i])
        for i in range(len(q_aligned))
    ])

    q_mean_lambda = karcher_mean_sphere(
        q_aligned,
        max_iter=100,
        tol=1e-6,
        verbose=True
    )

    distances = np.array([
        sphere_geodesic_distance(q_aligned[i], q_mean_lambda)
        for i in range(len(q_aligned))
    ])

    penalties = np.array([
        warping_penalty_integral(gammas[i])
        for i in range(len(gammas))
    ])

    summary = {
        "lambda_star": lambda_star,
        "mean_srvf_geodesic_distance": distances.mean(),
        "sd_srvf_geodesic_distance": distances.std(),
        "mean_warping_penalty": penalties.mean(),
        "sd_warping_penalty": penalties.std()
    }

    return q_mean_lambda, q_aligned, distances, penalties, summary

def validation_srvf_geodesic_mse_to_fixed_mean(q_mean, X_val_aligned, q_val_aligned=None):
    q_val = [
        normalize_srvf(srvf(X_val_aligned[i]))
        for i in range(len(X_val_aligned))
    ]

    if q_val_aligned is not None:
        q_val = q_val_aligned

    d2 = []

    for q in q_val:
        d = sphere_geodesic_distance(normalize_srvf(q), q_mean)
        d2.append(d ** 2)

    return float(np.mean(d2))

    
def invert_srvf_from_q(q, initial_point=None):
    """
    Inverse SRVF:
        X(t) = X(0) + integral_0^t q(s)||q(s)|| ds
    """
    if initial_point is None:
        initial_point = np.zeros(q.shape[1])

    vel = q * np.linalg.norm(q, axis=1, keepdims=True)

    X = np.zeros_like(q)
    X[0] = initial_point

    for j in range(1, len(q)):
        X[j] = X[j - 1] + 0.5 * (vel[j - 1] + vel[j]) * dt

    return X


def mean_pairwise_thickness(r_all):
    vals = []
    for i in range(len(r_all)):
        for j in range(i+1, len(r_all)):
            vals.append(l2_function_distance(r_all[i], r_all[j]))
    return np.mean(vals)

def restore_embedded_scale(curve, target_planar_length, target_center):
    """Calibrate the common inverse-SRVF scale using the planar length."""
    scale = target_planar_length / (curve_length_xy(curve) + 1e-8)
    return target_center + scale * (curve - curve.mean(axis=0))


def reconstruct_curves_from_srvfs(q_aligned, X_final):
    X_rec = []

    for i in range(len(q_aligned)):
        q_i = normalize_srvf(q_aligned[i])

        X_i = invert_srvf_from_q(
            q_i,
            initial_point=X_final[i, 0].copy()
        )

        X_i = enforce_curve_closure(X_i)

        X_i = restore_embedded_scale(
            X_i, curve_length_xy(X_final[i]), X_final[i].mean(axis=0)
        )

        X_i = enforce_curve_closure(X_i)
        X_rec.append(X_i)

    X_rec = np.array(X_rec)

    beta_rec = X_rec[:, :, :2]
    r_rec = X_rec[:, :, 2] / alpha

    beta_rec = np.array([
        enforce_curve_closure(beta_rec[i])
        for i in range(len(beta_rec))
    ])

    r_rec = np.array([
        enforce_scalar_periodicity(r_rec[i])
        for i in range(len(r_rec))
    ])

    X_rec = np.array([
        embed_curve(beta_rec[i], r_rec[i])
        for i in range(len(beta_rec))
    ])

    return X_rec, beta_rec, r_rec    

def mean_pairwise_shape(beta_all):
    vals = []
    for i in range(len(beta_all)):
        for j in range(i + 1, len(beta_all)):
            vals.append(l2_curve_distance(beta_all[i], beta_all[j]))
    return float(np.mean(vals))


def mean_pairwise_srvf_geodesic(X_all, q_all=None):
    if q_all is None:
        q_all = [normalize_srvf(srvf(X_all[i])) for i in range(len(X_all))]
    vals = []
    for i in range(len(q_all)):
        for j in range(i + 1, len(q_all)):
            vals.append(sphere_geodesic_distance(q_all[i], q_all[j]))
    return float(np.mean(vals))


def compute_srvf_tangent_fpca(q_aligned, mu, n_modes=3):
    return _srvf_numerics.tangent_fpca(
        [normalize_srvf(q) for q in q_aligned], mu, log_map_sphere,
        n_modes=n_modes, endpoint_duplicated=True
    )


def reconstruct_srvf_fpca_mode(mu, pc_flat, tau, initial_point=None):
    """
    Reconstruct plus/minus SRVF FPCA mode from tangent direction.
    """
    v = tau * pc_flat.reshape(mu.shape)

    q_mode = exp_map_sphere(v, mu)
    q_mode = normalize_srvf(q_mode)

    X_mode = invert_srvf_from_q(
        q_mode,
        initial_point=initial_point
    )

    X_mode = enforce_curve_closure(X_mode)

    beta_mode = X_mode[:, :2]
    r_mode = X_mode[:, 2] / alpha

    beta_mode = enforce_curve_closure(beta_mode)
    r_mode = enforce_scalar_periodicity(r_mode)

    return X_mode, beta_mode, r_mode

def gamma_to_psi(gamma, grid):
    gamma_dot = np.gradient(gamma, grid)
    gamma_dot = np.maximum(gamma_dot, 1e-8)
    psi = np.sqrt(gamma_dot)

    norm = np.sqrt(np.trapezoid(psi ** 2, grid))
    return psi / (norm + 1e-12)


def psi_to_gamma(psi, grid):
    density = psi ** 2
    gamma = np.zeros_like(grid)
    gamma[1:] = np.cumsum(
        0.5 * (density[:-1] + density[1:]) * np.diff(grid)
    )
    gamma = gamma / (gamma[-1] + 1e-12)
    gamma[0] = 0.0
    gamma[-1] = 1.0
    return gamma


def karcher_mean_gamma(gammas, grid):
    psi_all = np.array([
        gamma_to_psi(gammas[i], grid)
        for i in range(len(gammas))
    ])

    psi_all = psi_all[:, :, None]

    psi_mean = karcher_mean_sphere(
        psi_all,
        max_iter=100,
        tol=1e-6,
        verbose=False
    )

    psi_mean = psi_mean[:, 0]
    return psi_to_gamma(psi_mean, grid)


def srvf_sphere_geodesic_distance(q1, q2):
    """
    Geodesic distance on SRVF Hilbert sphere:
    d(q1, q2) = arccos(<q1, q2>)
    """
    inner = np.trapz(np.sum(q1 * q2, axis=1), dx=1.0 / (len(q1) - 1))

    # numerical stability
    inner = np.clip(inner, -1.0, 1.0)

    return np.arccos(inner)
    
# ============================================================
# 4. Synthetic data generation
# ============================================================

def phase_strength(mode):
    return {
        "mild": 0.06,
        "moderate": 0.12,
        "strong": 0.20,
        "very_strong": 0.30
    }.get(mode, 0.12)


def amp_strength(mode):
    return {
        "mild": 0.08,
        "moderate": 0.16,
        "strong": 0.25
    }.get(mode, 0.16)


def random_phase_warp(strength=0.20, n_modes=3):
    v = np.ones_like(s)

    for k in range(1, n_modes + 1):
        a = rng.normal(0.0, strength / k)
        b = rng.normal(0.0, strength / k)

        v += a * np.cos(2.0 * np.pi * k * s)
        v += b * np.sin(2.0 * np.pi * k * s)

    v = np.exp(v)

    gamma = np.cumsum(v)
    gamma = (gamma - gamma[0]) / (gamma[-1] - gamma[0])
    gamma = np.maximum.accumulate(np.clip(gamma, 0.0, 1.0))

    gamma[0] = 0.0
    gamma[-1] = 1.0

    return gamma


def flower_radius(u, n_petals=5, base_radius=1.0, amp=0.22, phase=0.0):
    theta = 2.0 * np.pi * u
    return base_radius * (1.0 + amp * np.cos(n_petals * theta + phase))


def curve_from_radius(u, rho, anisotropy=0.0, shear=0.0, rotate=0.0):
    theta = 2.0 * np.pi * u

    x = rho * np.cos(theta)
    y = rho * np.sin(theta)

    x = (1.0 + anisotropy) * x
    y = (1.0 - 0.5 * anisotropy) * y

    x2 = x + shear * y
    y2 = y

    c = np.cos(rotate)
    ss = np.sin(rotate)

    xr = c * x2 - ss * y2
    yr = ss * x2 + c * y2

    beta = np.column_stack([xr, yr])
    return ensure_ccw(beta)


def generate_single_curve():
    phs = phase_strength(PHASE_MODE)
    amps = amp_strength(AMPLITUDE_MODE)

    z = rng.normal(0.0, 1.0)
    gamma_true = random_phase_warp(strength=phs, n_modes=3)

    global_phase = rng.uniform(-np.pi, np.pi)
    anisotropy = rng.normal(0.0, 0.05)
    shear = rng.normal(0.0, 0.04)
    rotate = rng.normal(0.0, 0.15)

    amp_i = np.clip(0.20 * (1.0 + amps * z), 0.08, 0.32)

    rho = flower_radius(
        gamma_true,
        n_petals=5,
        amp=amp_i,
        phase=global_phase
    )

    beta = curve_from_radius(
        gamma_true,
        rho,
        anisotropy=anisotropy,
        shear=shear,
        rotate=rotate
    )

    if ADD_NOISE:
        beta += rng.normal(0.0, NOISE_LEVEL, size=beta.shape)
        beta[:, 0] = circular_smooth(beta[:, 0], 2)
        beta[:, 1] = circular_smooth(beta[:, 1], 2)
        
    beta = enforce_curve_closure(beta)
    
    return beta, gamma_true, z


def generate_coupled_thickness(beta_all, gamma_true_all, sim_seed=0):
    """
    Generate positive, smooth, phase-coupled thickness functions.

    Thickness is generated as a function of the same latent phase warp
    used to generate the inner curve. This makes thickness misalignment
    meaningful and allows alignment to improve thickness distance.
    """
    rng_local = np.random.default_rng(SEED + 1000 + sim_seed)
    r_all = np.zeros((len(beta_all), N))

    for i in range(len(beta_all)):
        beta = beta_all[i]
        u = gamma_true_all[i]

        kappa = compute_curvature_periodic(beta)
        unique_curvature = kappa[:-1]
        kappa_norm = (
            (kappa - unique_curvature.mean()) / (unique_curvature.std() + 1e-10)
        )

        phase_i = rng_local.uniform(0.0, 2.0 * np.pi)
        amp_i = rng_local.uniform(0.8, 1.4)

        # Keep the original draw count so subsequent subjects use the same RNG stream.
        noise = circular_smooth(
            rng_local.normal(size=N),
            n_iter=2
        )

        r = (
            0.20
            + amp_i * 0.040 * kappa_norm
            + amp_i * 0.035 * np.sin(2.0 * np.pi * u + phase_i)
            + amp_i * 0.020 * np.cos(4.0 * np.pi * u + 0.5 * phase_i)
            + 0.004 * noise
        )

        r = circular_smooth(r, n_iter=2)
        r = np.maximum(r, 0.08)
        r = enforce_scalar_periodicity(r)

        r_all[i] = r

    return r_all    
def generate_dataset(sim_id=0):
    beta_in_raw = np.zeros((K, N, 2))
    gamma_true = np.zeros((K, N))
    latent = np.zeros(K)

    for i in range(K):
        beta, gamma_i, z = generate_single_curve()
        beta_in_raw[i] = beta
        gamma_true[i] = gamma_i
        latent[i] = z

    r_raw = generate_coupled_thickness(beta_in_raw, gamma_true, sim_seed=sim_id)

    beta_out_raw = np.zeros_like(beta_in_raw)
    X_raw = np.zeros((K, N, 3))

    for i in range(K):
        beta_out_raw[i] = reconstruct_outer(beta_in_raw[i], r_raw[i])
        X_raw[i] = embed_curve(beta_in_raw[i], r_raw[i])

    return beta_in_raw, r_raw, beta_out_raw, X_raw, gamma_true, latent


# ============================================================
# 5. SRVF and Hilbert-sphere geometry
# ============================================================

def srvf(Y):
    dY = finite_difference_periodic(Y)
    speed = np.linalg.norm(dY, axis=1)
    speed = np.maximum(speed, 1e-12)
    return dY / np.sqrt(speed)[:, None]


def srvf_inner(q1, q2):
    return float(np.trapezoid(np.sum(q1*q2, axis=1), dx=dt))


def srvf_norm(q):
    return np.sqrt(max(srvf_inner(q, q), 0.0))


def normalize_srvf(q):
    nrm = srvf_norm(q)

    if nrm < 1e-12:
        return q.copy()

    return q / nrm


def compute_all_srvfs(X_all):
    q_raw = np.array([srvf(X_all[i]) for i in range(len(X_all))])
    q_hat = np.array([normalize_srvf(q_raw[i]) for i in range(len(X_all))])
    return q_raw, q_hat


def log_map_sphere(q, mu):
    ip = np.clip(srvf_inner(mu, q), -1.0, 1.0)
    theta = np.arccos(ip)

    if theta < 1e-10:
        return np.zeros_like(mu)

    v = q - ip * mu
    v_norm = srvf_norm(v)

    if v_norm < 1e-12:
        return np.zeros_like(mu)

    return (theta / v_norm) * v


def exp_map_sphere(v, mu):
    v_norm = srvf_norm(v)

    if v_norm < 1e-10:
        return mu.copy()

    return np.cos(v_norm) * mu + np.sin(v_norm) * (v / v_norm)


def sphere_geodesic_distance(q1, q2):
    ip = np.clip(srvf_inner(q1, q2), -1.0, 1.0)
    return np.arccos(ip)


def karcher_mean_sphere(q_list, max_iter=100, tol=1e-6, verbose=False):
    mu = normalize_srvf(q_list[0].copy())
    Kloc = len(q_list)

    for it in range(max_iter):
        tangent_mean = np.zeros_like(mu)

        for q in q_list:
            qn = normalize_srvf(q)
            tangent_mean += log_map_sphere(qn, mu)

        tangent_mean /= Kloc

        mu_new = exp_map_sphere(tangent_mean, mu)
        mu_new = normalize_srvf(mu_new)

        diff = srvf_norm(mu_new - mu)

        if verbose:
            print(f"Karcher iteration {it + 1}: diff = {diff:.3e}")

        if diff < tol:
            mu = mu_new
            break

        mu = mu_new

    return mu


def validation_srvf_geodesic_mse_to_karcher(X_train_aligned, X_val_aligned):
    """
    Validation error based on squared SRVF geodesic distance to
    the Hilbert-sphere Karcher mean of the aligned training SRVFs.
    """
    q_train = [
        normalize_srvf(srvf(X_train_aligned[i]))
        for i in range(len(X_train_aligned))
    ]

    mu = karcher_mean_sphere(q_train)

    d2 = []

    for i in range(len(X_val_aligned)):
        q_val = normalize_srvf(srvf(X_val_aligned[i]))
        d = sphere_geodesic_distance(q_val, mu)
        d2.append(d ** 2)

    return float(np.mean(d2)), mu


# ============================================================
# 6. Warping and rotation
# ============================================================

def interp_periodic(Y, gamma):
    """
    Interpolate closed curve/function on the closed grid [0,1].
    The endpoint is included and closure is enforced after interpolation.
    """
    Y = np.asarray(Y)
    gamma = np.asarray(gamma)

    grid = np.linspace(0.0, 1.0, len(Y))
    gamma = np.clip(gamma, 0.0, 1.0)

    if Y.ndim == 1:
        out = np.interp(gamma, grid, Y)
        out[-1] = out[0]
        return out

    out = np.zeros_like(Y)

    for j in range(Y.shape[1]):
        out[:, j] = np.interp(gamma, grid, Y[:, j])

    out[-1] = out[0]
    return out


def best_cyclic_shift(X_ref, X):
    best_shift = 0
    best_cost = np.inf

    for sh in range(N):
        Xs = _srvf_numerics.cyclic_shift(X, sh, True)
        cost = np.mean(np.sum((X_ref - Xs) ** 2, axis=1))

        if cost < best_cost:
            best_cost = cost
            best_shift = sh

    return best_shift


def optimal_rotation_2d(A, B):
    M = A.T @ B
    U, _, Vt = np.linalg.svd(M)

    R = U @ Vt

    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    return R

def apply_gamma_to_srvf(q, gamma):
    """
    Apply the SRVF reparameterization action

        (q, gamma)(s) = q(gamma(s)) * sqrt(gamma_dot(s)).
    """

    q = np.asarray(q)
    gamma = np.asarray(gamma)

    n, d = q.shape
    grid = np.linspace(0.0, 1.0, n)

    gamma = np.clip(gamma, 0.0, 1.0)

    q_warp = np.zeros_like(q)

    for j in range(d):
        q_warp[:, j] = np.interp(gamma, grid, q[:, j])

    gamma_dot = np.gradient(gamma, grid)
    gamma_dot = np.maximum(gamma_dot, 1e-8)

    q_warp = q_warp * np.sqrt(gamma_dot)[:, None]

    return normalize_srvf(q_warp)

def optimal_rotation_2d_from_srvf(q, q_ref):
    M = q[:, :2].T @ q_ref[:, :2]
    U, _, Vt = np.linalg.svd(M)
    R = U @ Vt

    if np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    return R

    
def dp_align_srvf(q_ref, q, lam=0.0):
    """Discrete quadrature approximation to L2 alignment plus lambda*P(gamma)."""
    return _srvf_numerics.penalized_dp(
        q_ref, q, lam, endpoint_duplicated=True
    )


def run_alignment(beta_raw, r_raw, X_raw, lambda_warp):
    """
    Full alignment:
        1. cyclic prealignment
        2. embedded SRVF warping
        3. SO(2) rotation of geometric coordinates

    Stores SRVF-stage diagnostics before final embedded closure correction.
    """
    Kloc = len(X_raw)

    beta_warped = beta_raw.copy()
    r_warped = r_raw.copy()
    X_warped = X_raw.copy()

    beta_aligned = beta_raw.copy()
    r_aligned = r_raw.copy()
    X_aligned = X_raw.copy()

    gammas = np.tile(np.linspace(0.0, 1.0, N), (Kloc, 1))

    X_template = X_raw[0].copy()
    q_template = normalize_srvf(srvf(X_template))

    q_warped_store = np.zeros_like(X_raw)
    q_aligned_store = np.zeros_like(X_raw)

    for it in range(N_ITER_ALIGN):
        print(f"      alignment iteration {it + 1}/{N_ITER_ALIGN}", flush=True)
        # Retain the actual sphere mean; its inverse is for cyclic initialization.

        for i in range(Kloc):
            beta_i = beta_raw[i].copy()
            r_i = r_raw[i].copy()
            X_i = X_raw[i].copy()

            # -----------------------------
            # 1. cyclic prealignment
            # -----------------------------
            if USE_CYCLIC_PREALIGNMENT:
                sh = best_cyclic_shift(X_template, X_i)

                beta_i = _srvf_numerics.cyclic_shift(beta_i, sh, True)
                r_i = _srvf_numerics.cyclic_shift(r_i, sh, True)

                beta_i = enforce_curve_closure(beta_i)
                r_i = enforce_scalar_periodicity(r_i)
                X_i = embed_curve(beta_i, r_i)

            # -----------------------------
            # 2. SRVF warping
            # -----------------------------
            q_i = normalize_srvf(srvf(X_i))
            gamma_i = dp_align_srvf(q_template, q_i, lam=lambda_warp)

            # -----------------------------
            # 2a. SRVF warping directly on Hilbert sphere
            # -----------------------------
            q_i_w = apply_gamma_to_srvf(q_i, gamma_i)
            q_i_w = normalize_srvf(q_i_w)

            # -----------------------------
            # 2b. Curve-domain warped objects only for display/evaluation
            # -----------------------------
            beta_i_w = interp_periodic(beta_i, gamma_i)
            r_i_w = interp_periodic(r_i, gamma_i)

            beta_i_w = enforce_curve_closure(beta_i_w)
            r_i_w = enforce_scalar_periodicity(r_i_w)

            X_i_w = embed_curve(beta_i_w, r_i_w)

            # -----------------------------
            # 3. SO(2) rotation directly in SRVF space
            # -----------------------------
            R_i = optimal_rotation_2d_from_srvf(q_i_w, q_template)

            q_i_a = q_i_w.copy()
            q_i_a[:, :2] = q_i_w[:, :2] @ R_i
            q_i_a = normalize_srvf(q_i_a)

            # For curve-domain output, rotate beta consistently for visualization/evaluation
            beta_i_a = beta_i_w @ R_i
            beta_i_a = enforce_curve_closure(beta_i_a)

            r_i_a = enforce_scalar_periodicity(r_i_w)

            X_i_a_preclosure = embed_curve(beta_i_a, r_i_a)

            # SRVF diagnostic after rotation: use SRVF-space aligned q, not srvf(X)
            q_i_a_preclosure = q_i_a.copy()

            # -----------------------------
            # 4. final geometric closure correction
            # -----------------------------
            X_i_a = enforce_curve_closure(X_i_a_preclosure)

            beta_i_a = X_i_a[:, :2]
            r_i_a = X_i_a[:, 2] / alpha

            # -----------------------------
            # Store outputs
            # -----------------------------
            beta_warped[i] = beta_i_w
            r_warped[i] = r_i_w
            X_warped[i] = X_i_w

            beta_aligned[i] = beta_i_a
            r_aligned[i] = r_i_a
            X_aligned[i] = X_i_a

            q_warped_store[i] = q_i_w
            q_aligned_store[i] = q_i_a_preclosure

            gammas[i] = gamma_i

        q_template = karcher_mean_sphere(
            q_aligned_store,
            max_iter=100,
            tol=1e-6,
            verbose=True
        )
        X_template = invert_srvf_from_q(q_template)
        X_template = enforce_curve_closure(X_template)

    return (
        beta_warped, r_warped, X_warped,
        beta_aligned, r_aligned, X_aligned,
        gammas,
        q_warped_store, q_aligned_store,
        q_template, X_template
    )


def align_to_fixed_template(beta_raw, r_raw, X_raw, X_template, lambda_warp, q_reference=None, return_srvfs=False):
    """
    Align curves to a fixed embedded template.
    """
    Kloc = len(X_raw)

    beta_warped = beta_raw.copy()
    r_warped = r_raw.copy()
    X_warped = X_raw.copy()

    beta_aligned = beta_raw.copy()
    r_aligned = r_raw.copy()
    X_aligned = X_raw.copy()

    gammas = np.tile(np.linspace(0.0, 1.0, N), (Kloc, 1))

    q_template = normalize_srvf(srvf(X_template))
    if q_reference is not None:
        q_template = normalize_srvf(np.asarray(q_reference).copy())
    q_store = np.zeros_like(X_raw)

    for i in range(Kloc):
        
        if i % 25 == 0:
            print(f"        fixed-template aligned {i}/{Kloc}", flush=True)

        beta_i = beta_raw[i].copy()
        r_i = r_raw[i].copy()
        X_i = X_raw[i].copy()

        if USE_CYCLIC_PREALIGNMENT:
            sh = best_cyclic_shift(X_template, X_i)

            beta_i = _srvf_numerics.cyclic_shift(beta_i, sh, True)
            r_i = _srvf_numerics.cyclic_shift(r_i, sh, True)

            beta_i = enforce_curve_closure(beta_i)
            r_i = enforce_scalar_periodicity(r_i)
            X_i = embed_curve(beta_i, r_i)

        q_i = normalize_srvf(srvf(X_i))
        gamma_i = dp_align_srvf(q_template, q_i, lam=lambda_warp)

        beta_i_w = interp_periodic(beta_i, gamma_i)
        r_i_w = interp_periodic(r_i, gamma_i)

        beta_i_w = enforce_curve_closure(beta_i_w)
        r_i_w = enforce_scalar_periodicity(r_i_w)

        X_i_w = embed_curve(beta_i_w, r_i_w)

        q_i_w = apply_gamma_to_srvf(q_i, gamma_i)
        q_i_w = normalize_srvf(q_i_w)

        R_i = optimal_rotation_2d_from_srvf(q_i_w, q_template)

        q_i_a = q_i_w.copy()
        q_i_a[:, :2] = q_i_w[:, :2] @ R_i
        q_i_a = normalize_srvf(q_i_a)

        beta_i_a = beta_i_w @ R_i
        beta_i_a = enforce_curve_closure(beta_i_a)

        r_i_a = enforce_scalar_periodicity(r_i_w)

        X_i_a = embed_curve(beta_i_a, r_i_a)

        # Final embedded closure correction:
        # enforce X_i^a(1) = X_i^a(0) once, and do not re-embed afterward.
        X_i_a = enforce_curve_closure(X_i_a)

        beta_i_a = X_i_a[:, :2]
        r_i_a = enforce_scalar_periodicity(X_i_a[:, 2] / alpha)

        beta_warped[i] = beta_i_w
        r_warped[i] = r_i_w
        X_warped[i] = X_i_w

        beta_aligned[i] = beta_i_a
        r_aligned[i] = r_i_a
        X_aligned[i] = X_i_a

        gammas[i] = gamma_i
        q_store[i] = q_i_a

    result = beta_warped, r_warped, X_warped, beta_aligned, r_aligned, X_aligned, gammas
    return result + (q_store,) if return_srvfs else result


# ============================================================
# 7. Lambda selection by K-fold CV
# ============================================================

def select_lambda_5fold_cv(beta_raw, r_raw, X_raw, sim_id=0):
    """
    Select lambda by K-fold CV.

    For each lambda and fold:
        1. Align training curves.
        2. Compute aligned training template.
        3. Align validation curves to the template.
        4. Compute Karcher mean of aligned training SRVFs.
        5. Compute validation SRVF geodesic MSE to that mean.
    """
    rows = []

    kf = KFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE + sim_id
    )

    for lam in LAMBDA_GRID:
        print(f"    lambda={lam}")

        for fold_id, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(X_raw)))):
            print(f"        fold {fold_id + 1}/{N_FOLDS}")

            beta_train = beta_raw[train_idx]
            r_train = r_raw[train_idx]
            X_train = X_raw[train_idx]

            beta_val = beta_raw[val_idx]
            r_val = r_raw[val_idx]
            X_val = X_raw[val_idx]

            train_alignment = run_alignment(
                beta_train,
                r_train,
                X_train,
                lambda_warp=lam
            )

            (
                beta_train_w,
                r_train_w,
                X_train_w,
                beta_train_a,
                r_train_a,
                X_train_a,
                gam_train,
                q_train_w,
                q_train_a,
                q_mean,
                X_template
            ) = train_alignment

           

            val_alignment = align_to_fixed_template(
                beta_val,
                r_val,
                X_val,
                X_template=X_template,
                lambda_warp=lam, q_reference=q_mean, return_srvfs=True
            )

            (
                beta_val_w,
                r_val_w,
                X_val_w,
                beta_val_a,
                r_val_a,
                X_val_a,
                gam_val, q_val_a
            ) = val_alignment

            srvf_geo_mse = validation_srvf_geodesic_mse_to_fixed_mean(
                q_mean=q_mean,
                X_val_aligned=X_val_a, q_val_aligned=q_val_a
            )

            rows.append({
                "sim": sim_id,
                "lambda": lam,
                "fold": fold_id,
                "srvf_geodesic_validation_mse": srvf_geo_mse,
                "mean_gamma_deviation_mse": gamma_deviation_mse(gam_val)
            })

    cv_table = pd.DataFrame(rows)

    cv_summary = (
        cv_table
        .groupby("lambda")["srvf_geodesic_validation_mse"]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={
            "mean": "cv_srvf_geodesic_mse_mean",
            "std": "cv_srvf_geodesic_mse_sd"
        })
    )

    best_idx = cv_summary["cv_srvf_geodesic_mse_mean"].idxmin()
    best_lambda = cv_summary.loc[best_idx, "lambda"]

    print(
        "    selected lambda =",
        best_lambda,
        "| CV SRVF geodesic MSE =",
        cv_summary.loc[best_idx, "cv_srvf_geodesic_mse_mean"]
    )

    cv_table["best_lambda"] = best_lambda
    cv_summary["best_lambda"] = best_lambda

    return best_lambda, cv_table, cv_summary


def gamma_deviation_mse(gammas):
    identity = np.linspace(0.0, 1.0, gammas.shape[1])
    return float(np.mean((gammas - identity[None, :]) ** 2))


def curve_length_xy(curve):
    """
    Curve length using only XY coordinates.
    Works for both (N,2) and (N,3) curves.
    """
    if curve.shape[1] == 3:
        C = curve[:, :2]
    else:
        C = curve

    dC = np.diff(C, axis=0)
    return float(np.sum(np.linalg.norm(dC, axis=1)))
    
# ============================================================
# 8. Evaluation
# ============================================================

def l2_curve_distance(A, B):
    return np.sqrt(np.mean(np.sum((A - B) ** 2, axis=1)))


def l2_function_distance(f, g):
    return np.sqrt(np.mean((f - g) ** 2))


def evaluate_alignment_to_template(
    beta_raw,
    r_raw,
    X_raw,
    beta_aligned,
    r_aligned,
    X_aligned,
    X_template, q_reference=None, q_aligned=None
):
    beta_template = X_template[:, :2]
    r_template = X_template[:, 2] / alpha
    q_template = normalize_srvf(srvf(X_template)) if q_reference is None else normalize_srvf(q_reference)

    shape_before = []
    shape_after = []
    thick_before = []
    thick_after = []
    srvf_before = []
    srvf_after = []

    for i in range(len(beta_raw)):
        shape_before.append(l2_curve_distance(beta_raw[i], beta_template))
        shape_after.append(l2_curve_distance(beta_aligned[i], beta_template))

        thick_before.append(l2_function_distance(r_raw[i], r_template))
        thick_after.append(l2_function_distance(r_aligned[i], r_template))

        q_raw_i = normalize_srvf(srvf(X_raw[i]))
        q_aligned_i = normalize_srvf(srvf(X_aligned[i])) if q_aligned is None else normalize_srvf(q_aligned[i])

        srvf_before.append(sphere_geodesic_distance(q_raw_i, q_template))
        srvf_after.append(sphere_geodesic_distance(q_aligned_i, q_template))

    return pd.DataFrame([
        {
            "Metric": "Mean shape distance",
            "Before alignment": np.mean(shape_before),
            "After alignment": np.mean(shape_after),
            "Change": 100 * (np.mean(shape_before) - np.mean(shape_after)) / np.mean(shape_before)
        },
        {
            "Metric": "Mean thickness distance",
            "Before alignment": np.mean(thick_before),
            "After alignment": np.mean(thick_after),
            "Change": 100 * (np.mean(thick_before) - np.mean(thick_after)) / np.mean(thick_before)
        },
        {
            "Metric": "Mean SRVF geodesic distance",
            "Before alignment": np.mean(srvf_before),
            "After alignment": np.mean(srvf_after),
            "Change": 100 * (np.mean(srvf_before) - np.mean(srvf_after)) / np.mean(srvf_before)
        }
    ])


def summarize_repeated_results(df, group_cols, metric_cols):
    rows = []

    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        for m in metric_cols:
            row[f"{m}_mean"] = g[m].mean()
            row[f"{m}_sd"] = g[m].std()

        rows.append(row)

    return pd.DataFrame(rows)

def mean_endpoint_error(curves):
    """
    Mean endpoint closure error over a collection of curves.
    """
    return float(np.mean([
        endpoint_closure_error(curves[i])
        for i in range(len(curves))
    ]))


def fpca_reconstruct_from_flat(mean_flat, pcs, scores, n_modes):
    return mean_flat + scores[:, :n_modes] @ pcs[:n_modes]


def compare_aligned_fpca_vs_srvf_tangent_fpca(
    beta_aligned,
    r_aligned,
    X_final,
    srvf_fpca_payload,
    n_modes_list=[1, 2, 3]
):
    """
    Compare ordinary aligned-domain FPCA with SRVF tangent-space FPCA.

    Errors are computed in reconstructed curve/thickness domain.
    """

    # ---------- ordinary aligned FPCA ----------
    beta_mean_flat, beta_pcs, beta_scores, _ = compute_fpca_modes(beta_aligned)
    r_mean_flat, r_pcs, r_scores, _ = compute_fpca_modes(r_aligned)

    # ---------- SRVF tangent FPCA ----------
    mu = srvf_fpca_payload["mu"]
    pcs_srvf = srvf_fpca_payload["pcs"]
    scores_srvf = srvf_fpca_payload["scores"]

    rows = []

    for m in n_modes_list:

        # ====================================================
        # A. Ordinary aligned-domain FPCA reconstruction
        # ====================================================
        beta_hat_flat = fpca_reconstruct_from_flat(
            beta_mean_flat,
            beta_pcs,
            beta_scores,
            n_modes=m
        )

        beta_hat = beta_hat_flat.reshape(len(beta_aligned), N, 2)

        r_hat = fpca_reconstruct_from_flat(
            r_mean_flat,
            r_pcs,
            r_scores,
            n_modes=m
        )

        shape_err_fpca = np.mean([
            l2_curve_distance(beta_aligned[i], beta_hat[i])
            for i in range(len(beta_aligned))
        ])

        thick_err_fpca = np.mean([
            l2_function_distance(r_aligned[i], r_hat[i])
            for i in range(len(r_aligned))
        ])

        # ====================================================
        # B. SRVF tangent-space FPCA reconstruction
        # ====================================================
        beta_hat_srvf = []
        r_hat_srvf = []

        for i in range(len(X_final)):
            v_hat_flat = scores_srvf[i, :m] @ pcs_srvf[:m]
            v_hat = v_hat_flat.reshape(mu.shape)

            q_hat = exp_map_sphere(v_hat, mu)
            q_hat = normalize_srvf(q_hat)

            X_hat = invert_srvf_from_q(
                q_hat,
                initial_point=X_final[i, 0].copy()
            )

            X_hat = enforce_curve_closure(X_hat)

            X_hat = restore_embedded_scale(
                X_hat, curve_length_xy(X_final[i]), X_final[i].mean(axis=0)
            )

            beta_hat_srvf.append(enforce_curve_closure(X_hat[:, :2]))
            r_hat_srvf.append(enforce_scalar_periodicity(X_hat[:, 2] / alpha))

        beta_hat_srvf = np.array(beta_hat_srvf)
        r_hat_srvf = np.array(r_hat_srvf)

        shape_err_srvf = np.mean([
            l2_curve_distance(beta_aligned[i], beta_hat_srvf[i])
            for i in range(len(beta_aligned))
        ])

        thick_err_srvf = np.mean([
            l2_function_distance(r_aligned[i], r_hat_srvf[i])
            for i in range(len(r_aligned))
        ])

        rows.append({
            "n_modes": m,

            "shape_error_aligned_fpca": shape_err_fpca,
            "shape_error_srvf_tangent_fpca": shape_err_srvf,
            "shape_improvement_percent": 100 * (shape_err_fpca - shape_err_srvf) / shape_err_fpca,

            "thickness_error_aligned_fpca": thick_err_fpca,
            "thickness_error_srvf_tangent_fpca": thick_err_srvf,
            "thickness_improvement_percent": 100 * (thick_err_fpca - thick_err_srvf) / thick_err_fpca
        })

    return pd.DataFrame(rows)


def compare_srvf_tangent_fpca_reconstruction(q_aligned, srvf_fpca_payload, n_modes_list=[1, 2, 3]):
    """
    Reconstruction error of SRVF tangent-space FPCA measured in SRVF geodesic distance.
    This is the correct metric for SRVF tangent FPCA.
    """
    mu = srvf_fpca_payload["mu"]
    pcs = srvf_fpca_payload["pcs"]
    scores = srvf_fpca_payload["scores"]

    rows = []

    for m in n_modes_list:
        errors = []

        for i in range(len(q_aligned)):
            v_hat_flat = scores[i, :m] @ pcs[:m]
            v_hat = v_hat_flat.reshape(mu.shape)

            q_hat = exp_map_sphere(v_hat, mu)
            q_hat = normalize_srvf(q_hat)

            q_true = normalize_srvf(q_aligned[i])

            errors.append(sphere_geodesic_distance(q_true, q_hat))

        rows.append({
            "n_modes": m,
            "srvf_tangent_fpca_geodesic_error": float(np.mean(errors)),
            "srvf_tangent_fpca_geodesic_error_sd": float(np.std(errors))
        })

    return pd.DataFrame(rows)

def srvf_mean_only_error(q_aligned, mu):
    errors = []
    for i in range(len(q_aligned)):
        q_i = normalize_srvf(q_aligned[i])
        errors.append(sphere_geodesic_distance(q_i, mu))
    return float(np.mean(errors))
    
# ============================================================
# 9. Figures
# ============================================================

def plot_lambda_frequency(lambda_summary_results):
    df = lambda_summary_results[["sim", "best_lambda"]].drop_duplicates()
    counts = df["best_lambda"].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.bar(counts.index.astype(str), counts.values)
    ax.set_xlabel(r"Selected $\lambda$")
    ax.set_ylabel("Frequency")
    ax.set_title("Selected lambda distribution across simulations")

    plt.tight_layout()

    save_figure(
        fig,
        "fig_lambda_frequency",
        caption=(
            "Distribution of selected warping regularization parameters "
            "across repeated synthetic simulations."
        ),
        label="fig:synthetic_lambda_frequency"
    )

    plt.close(fig)


def plot_lambda_curve_repeated(lambda_results):
    summary = (
        lambda_results
        .groupby("lambda")["srvf_geodesic_validation_mse"]
        .agg(["mean", "std"])
        .reset_index()
    )

    x = summary["lambda"].values.copy()
    x[x == 0] = 1e-4

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.semilogx(
        x,
        summary["mean"],
        marker="o",
        lw=2,
        label="Mean K-fold CV SRVF geodesic MSE"
    )

    ax.fill_between(
        x,
        summary["mean"] - summary["std"],
        summary["mean"] + summary["std"],
        alpha=0.25,
        label=r"$\pm 1$ SD"
    )

    ax.set_xlabel(r"$\lambda$")
    ax.set_ylabel("SRVF geodesic validation MSE")
    ax.set_title("K-fold CV lambda selection curve")
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        "fig_lambda_curve_repeated",
        caption=(
            "Five-fold cross-validation curve for selecting the warping "
            "regularization parameter using SRVF geodesic validation error."
        ),
        label="fig:synthetic_lambda_curve_repeated"
    )

    plt.close(fig)


def plot_pipeline_example(
    beta_raw,
    beta_out_raw,
    q_raw,
    q_hat,
    beta_warped,
    beta_aligned,
    r_aligned,
    gammas,
    X_template,
    srvf_r_reconstructed=None
):
    Kloc = len(beta_raw)

    beta_mean = enforce_curve_closure(X_template[:, :2])
    r_mean = enforce_scalar_periodicity(X_template[:, 2] / alpha)
    beta_out_mean = reconstruct_outer(beta_mean, r_mean)

    q_raw_mag = np.linalg.norm(q_raw, axis=2)
    q_hat_mag = np.linalg.norm(q_hat, axis=2)

    q_raw_mean = q_raw_mag.mean(axis=0)
    q_raw_sd = q_raw_mag.std(axis=0)

    q_hat_mean = q_hat_mag.mean(axis=0)
    q_hat_sd = q_hat_mag.std(axis=0)

    # for Step 3
    identity = np.linspace(0.0, 1.0, gammas.shape[1])
    gamma_dev = gammas - identity[None, :]

    gamma_karcher_mean = karcher_mean_gamma(gammas, identity)
    gamma_karcher_dev = gamma_karcher_mean - identity
    gamma_sd = gamma_dev.std(axis=0)

    # --------------------------------------------------------
    # Colors
    # --------------------------------------------------------
    col_inner = "#1f5fd6"
    col_outer = "#d62728"
    col_mean = "#111111"
    col_thick = "#1f5fd6"
    col_srvf = "#174a7e"
    col_norm = "#d95f02"
    col_warp = "#1b7f2a"
    col_after_warp = "#7b1fa2"
    col_aligned = "#8b0000"
    col_gray = "0.72"

    fig, axes = plt.subplots(3, 3, figsize=(17, 13.5))

    def clean_axis(ax, equal=False):
        ax.grid(True, linestyle="--", alpha=0.20)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if equal:
            ax.axis("equal")
            ax.set_xlabel(r"$\beta_x(s)$")
            ax.set_ylabel(r"$\beta_y(s)$")


    # ========================================================
    # Step 1: raw synthetic contours
    # ========================================================
    ax = axes[0, 0]

    for i in range(Kloc):

        B_in = close_for_plot(beta_raw[i])
        B_out = close_for_plot(beta_out_raw[i])

        ax.plot(
            B_in[:, 0],
            B_in[:, 1],
            color=col_inner,
            alpha=0.50,
            lw=1.0
        )

        ax.plot(
            B_out[:, 0],
            B_out[:, 1],
            color=col_outer,
            alpha=0.50,
            lw=1.0
        )

    ax.set_title("Raw ventricular contours")
    ax.set_xlabel(r"$\beta_x(s)$")
    ax.set_ylabel(r"$\beta_y(s)$")

    clean_axis(ax, equal=True)

    inner_handle = Line2D(
        [0], [0],
        color=col_inner,
        lw=2.5,
        alpha=1.0,
        label=r"Raw inner contours $\beta_i(s)$"
    )

    outer_handle = Line2D(
        [0], [0],
        color=col_outer,
        lw=2.5,
        alpha=1.0,
        label=r"Raw outer contours $\beta_i^{\mathrm{out}}(s)$"
    )

    ax.legend(
        handles=[inner_handle, outer_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 2: embedded curves, XY projection
    # ========================================================
    ax = axes[0, 1]

    for i in range(Kloc):

        B = close_for_plot(beta_raw[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_gray,
            alpha=0.50,
            lw=0.9
        )

    B_template = close_for_plot(beta_mean)

    ax.plot(
        B_template[:, 0],
        B_template[:, 1],
        color=col_mean,
        lw=3.0
    )

    ax.set_title(r"Embedded curve projections")
    ax.set_xlabel(r"$\beta_x(s)$")
    ax.set_ylabel(r"$\beta_y(s)$")

    clean_axis(ax, equal=True)

    raw_proj_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        alpha=1.0,
        label=r"Raw embedded projections"
    )

    template_handle = Line2D(
        [0], [0],
        color=col_mean,
        lw=3.0,
        alpha=1.0,
        label=r"SRVF Karcher mean projection $\bar{\beta}(s)$"
    )

    ax.legend(
        handles=[raw_proj_handle, template_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 3: warping deviation from identity
    # ========================================================
    ax = axes[0, 2]

    for i in range(Kloc):

        ax.plot(
            s,
            gamma_dev[i],
            color=col_gray,
            alpha=0.50,
            lw=0.9
        )

    ax.fill_between(
        s,
        gamma_karcher_dev - 2 * gamma_sd,
        gamma_karcher_dev + 2 * gamma_sd,
        color=col_warp,
        alpha=0.16
    )

    ax.plot(
        s,
        gamma_karcher_dev,
        color=col_warp,
        lw=3.0
    )

    ax.axhline(
        0.0,
        color="black",
        linestyle="--",
        lw=1.6
    )

    ax.set_title(r"Warping deviations from identity")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$\gamma_i(s)-s$")

    clean_axis(ax)

    warp_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        alpha=1.0,
        label=r"Individual warping deviations"
    )

    warp_sd_handle = Line2D(
        [0], [0],
        color=col_warp,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    warp_mean_handle = Line2D(
        [0], [0],
        color=col_warp,
        lw=3.0,
        alpha=1.0,
        label=r"Karcher mean deviation"
    )

    ax.legend(
        handles=[warp_handle, warp_sd_handle, warp_mean_handle],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    max_dev = np.nanmax(np.abs(gamma_dev))

    if np.isfinite(max_dev) and max_dev > 0:

        ax.set_ylim(
            -1.15 * max_dev,
             1.15 * max_dev
        )

    # ========================================================
    # Step 4: raw SRVF magnitude
    # ========================================================
    ax = axes[1, 0]

    for i in range(Kloc):

        ax.plot(
            s,
            q_raw_mag[i],
            color=col_gray,
            alpha=0.50,
            lw=0.75
        )

    ax.fill_between(
        s,
        q_raw_mean - 2 * q_raw_sd,
        q_raw_mean + 2 * q_raw_sd,
        color=col_srvf,
        alpha=0.12
    )

    ax.plot(
        s,
        q_raw_mean,
        color=col_srvf,
        lw=3.0
    )

    ax.set_title(r"Raw SRVF magnitudes")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$\|q_i(s)\|$")

    clean_axis(ax)

    raw_srvf_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        alpha=1.0,
        label=r"Raw SRVF magnitudes"
    )

    raw_srvf_sd_handle = Line2D(
        [0], [0],
        color=col_srvf,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    raw_srvf_mean_handle = Line2D(
        [0], [0],
        color=col_srvf,
        lw=3.0,
        alpha=1.0,
        label=r"Pointwise mean magnitude"
    )

    ax.legend(
        handles=[raw_srvf_handle, raw_srvf_sd_handle, raw_srvf_mean_handle],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    # ========================================================
    # Step 5: normalized SRVF magnitude
    # ========================================================
    ax = axes[1, 1]

    for i in range(Kloc):

        ax.plot(
            s,
            q_hat_mag[i],
            color=col_gray,
            alpha=0.50,
            lw=0.75
        )

    ax.fill_between(
        s,
        q_hat_mean - 2 * q_hat_sd,
        q_hat_mean + 2 * q_hat_sd,
        color=col_norm,
        alpha=0.14
    )

    ax.plot(
        s,
        q_hat_mean,
        color=col_norm,
        lw=3.0
    )

    ax.set_title(r"Normalized SRVF magnitudes")
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$\|\tilde q_i(s)\|$")

    clean_axis(ax)

    norm_srvf_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        alpha=1.0,
        label=r"Normalized SRVF magnitudes"
    )

    norm_srvf_sd_handle = Line2D(
        [0], [0],
        color=col_norm,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    norm_srvf_mean_handle = Line2D(
        [0], [0],
        color=col_norm,
        lw=3.0,
        alpha=1.0,
        label=r"Pointwise mean magnitude"
    )

    ax.legend(
        handles=[norm_srvf_handle, norm_srvf_sd_handle, norm_srvf_mean_handle],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    # ========================================================
    # Step 6: SRVF-aligned thickness component
    # ========================================================
    ax = axes[1, 2]

    r_display = (
        r_aligned
        if srvf_r_reconstructed is None
        else srvf_r_reconstructed
    )

    for i in range(Kloc):

        ax.plot(
            s,
            r_display[i],
            color=col_gray,
            alpha=0.50,
            lw=0.9
        )

    r_karcher = enforce_scalar_periodicity(
        X_template[:, 2] / alpha
    )

    r_sd = r_display.std(axis=0)

    ax.fill_between(
        s,
        r_karcher - 2 * r_sd,
        r_karcher + 2 * r_sd,
        color=col_thick,
        alpha=0.14
    )

    ax.plot(
        s,
        r_karcher,
        color=col_thick,
        lw=3.0
    )

    ax.set_title(r"SRVF-aligned thickness component"
)
    ax.set_xlabel(r"$s$")
    ax.set_ylabel(r"$r_{\mathrm{SRVF}}(s)$")

    clean_axis(ax)

    thick_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.5,
        alpha=1.0,
        label=r"Aligned thickness profiles $r_{\mathrm{SRVF},i}(s)$"
    )

    thick_sd_handle = Line2D(
        [0], [0],
        color=col_thick,
        lw=6,
        alpha=0.25,
        label=r"Pointwise $\pm 2$ SD"
    )

    thick_mean_handle = Line2D(
        [0], [0],
        color=col_thick,
        lw=3.0,
        alpha=1.0,
        label=r"Karcher mean thickness $\bar r_{\mathrm{SRVF}}(s)$"
    )

    ax.legend(
        handles=[thick_handle, thick_sd_handle, thick_mean_handle],
        frameon=True,
        fontsize=9,
        loc="upper right"
    )

    # ========================================================
    # Step 7: after embedded warping
    # ========================================================
    ax = axes[2, 0]

    for i in range(Kloc):

        B = close_for_plot(beta_warped[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_after_warp,
            alpha=0.50,
            lw=0.85
        )

    ax.set_title(r"Contours after embedded warping")
    ax.set_xlabel(r"$\beta_x(s)$")
    ax.set_ylabel(r"$\beta_y(s)$")

    clean_axis(ax, equal=True)

    warped_handle = Line2D(
        [0], [0],
        color=col_after_warp,
        lw=2.5,
        alpha=1.0,
        label=r"Warped contours $\beta_i(\gamma_i(s))$"
    )

    ax.legend(
        handles=[warped_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 8: after SO(2) rotation
    # ========================================================
    ax = axes[2, 1]

    for i in range(Kloc):

        B = close_for_plot(beta_aligned[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_aligned,
            alpha=0.50,
            lw=0.85
        )

    ax.set_title(r"Contours after $SO(2)$ alignment")
    ax.set_xlabel(r"$\beta_x(s)$")
    ax.set_ylabel(r"$\beta_y(s)$")

    clean_axis(ax, equal=True)

    aligned_handle = Line2D(
        [0], [0],
        color=col_aligned,
        lw=2.5,
        alpha=1.0,
        label=r"Aligned contours $R_i(\beta_i(\gamma_i(s)))$"
    )

    ax.legend(
        handles=[aligned_handle],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    # ========================================================
    # Step 9: SRVF Karcher mean reconstruction
    # ========================================================
    ax = axes[2, 2]

    for i in range(Kloc):

        B = close_for_plot(beta_aligned[i])

        ax.plot(
            B[:, 0],
            B[:, 1],
            color=col_gray,
            alpha=0.50,
            lw=0.75
        )

    B_in = close_for_plot(beta_mean)
    B_out = close_for_plot(beta_out_mean)

    poly_x = np.concatenate([
        B_out[:, 0],
        B_in[::-1, 0]
    ])

    poly_y = np.concatenate([
        B_out[:, 1],
        B_in[::-1, 1]
    ])

    ax.fill(
        poly_x,
        poly_y,
        color="0.80",
        alpha=0.55
    )

    ax.plot(
        B_in[:, 0],
        B_in[:, 1],
        color=col_inner,
        lw=3.2
    )

    ax.plot(
        B_out[:, 0],
        B_out[:, 1],
        color=col_outer,
        linestyle="--",
        lw=3.0
    )

    ax.set_title(r"SRVF Karcher mean reconstruction")
    ax.set_xlabel(r"$\beta_x(s)$")
    ax.set_ylabel(r"$\beta_y(s)$")

    clean_axis(ax, equal=True)

    band_handle = Line2D(
        [0], [0],
        color="0.60",
        lw=6,
        alpha=0.55,
        label=r"Mean thickness band"
    )

    mean_inner_handle = Line2D(
        [0], [0],
        color=col_inner,
        lw=3.2,
        alpha=1.0,
        label=r"Mean inner contour $\bar{\beta}(s)$"
    )

    mean_outer_handle = Line2D(
        [0], [0],
        color=col_outer,
        linestyle="--",
        lw=3.0,
        alpha=1.0,
        label=r"Mean outer contour $\bar{\beta}_{\mathrm{out}}(s)$"
    )

    raw_aligned_handle = Line2D(
        [0], [0],
        color=col_gray,
        lw=2.0,
        alpha=1.0,
        label=r"Aligned contours"
    )    

    ax.legend(
        handles=[
            raw_aligned_handle,
            band_handle,
            mean_inner_handle,
            mean_outer_handle
        ],
        frameon=True,
        fontsize=9,
        loc="lower left"
    )

    plt.tight_layout()

    save_figure(
        fig,
        "fig_synthetic_pipeline_example",
        caption=(
            "Representative synthetic embedded SRVF alignment pipeline. "
            "The raw inner and outer contours are shown with consistent colors. "
            "The warping parameter was selected by cross-validation using SRVF "
            "geodesic validation error. The final panel shows the reconstructed "
            "mean inner curve, mean outer curve, and the mean thickness band."
        ),
        label="fig:synthetic_pipeline_example"
    )

    plt.close(fig)


def compute_fpca_modes(Y):
    if Y.ndim == 3:
        Kloc, Nloc, d = Y.shape
        Y_flat = Y.reshape(Kloc, Nloc * d)
    else:
        Y_flat = Y

    mean_flat = Y_flat.mean(axis=0)
    Y_centered = Y_flat - mean_flat

    U, S, Vt = np.linalg.svd(Y_centered, full_matrices=False)

    scores = U * S
    pcs = Vt

    return mean_flat, pcs, scores, S


def plot_thickness_fpca_modes(r_aligned, name="fig_synthetic_thickness_fpca_modes"):
    r_mean, pcs, scores, _ = compute_fpca_modes(r_aligned)

    n_modes = min(3, pcs.shape[0])
    fig, axes = plt.subplots(n_modes, 1, figsize=(9, 4 * n_modes), sharex=True)

    if n_modes == 1:
        axes = [axes]

    for k in range(n_modes):
        pc = pcs[k]
        sd_score = np.std(scores[:, k])

        r_minus = r_mean - 2 * sd_score * pc
        r_plus = r_mean + 2 * sd_score * pc

        ax = axes[k]

        for i in range(len(r_aligned)):
            ax.plot(s, r_aligned[i], color="lightgray", alpha=0.35, lw=1)

        ax.plot(s, r_mean, "k", lw=3, label="Mean thickness")
        ax.plot(s, r_minus, "--", lw=2.5, label="-2 SD")
        ax.plot(s, r_plus, "--", lw=2.5, label="+2 SD")

        ax.set_title(f"Thickness FPCA mode {k + 1}")
        ax.set_ylabel("Thickness")
        ax.legend(loc="upper left")

    axes[-1].set_xlabel("Normalized parameter")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Functional principal component modes of the aligned thickness functions. "
            "Dashed curves show plus and minus two standard deviations along each mode."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)


    
def plot_shape_fpca_modes(beta_aligned, r_aligned, name="fig_synthetic_shape_fpca_modes"):
    """
    Clean FPCA visualization (Q1-ready)

    - Inner curves only (outer removed for clarity)
    - Fixed axis limits
    - Minimal legend
    """

    beta_mean_flat, pcs, scores, _ = compute_fpca_modes(beta_aligned)
    beta_mean = beta_mean_flat.reshape(N, 2)

    n_modes = min(3, pcs.shape[0])

    fig, axes = plt.subplots(1, n_modes, figsize=(5 * n_modes, 5))

    if n_modes == 1:
        axes = [axes]

    # global axis limits (stable across modes)
    lim = 1.6

    for k in range(n_modes):
        pc = pcs[k].reshape(N, 2)
        sd_score = np.std(scores[:, k])

        beta_minus = beta_mean - 2 * sd_score * pc
        beta_plus  = beta_mean + 2 * sd_score * pc

        ax = axes[k]

        # background curves (light)
        for i in range(len(beta_aligned)):
            B = close_for_plot(beta_aligned[i])
            ax.plot(B[:, 0], B[:, 1], color="lightgray", alpha=0.25, lw=0.8)

        # mean + modes
        ax.plot(*close_for_plot(beta_mean).T,
                color="black", lw=3, linestyle="-", label="Mean")

        ax.plot(*close_for_plot(beta_minus).T,
                color="#1f77b4", lw=2.5, linestyle="--", label="-2 SD")

        ax.plot(*close_for_plot(beta_plus).T,
                color="#d95f02", lw=2.5, linestyle="--", label="+2 SD")

        ax.set_title(f"FPCA mode {k + 1}")

        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")

        ax.grid(True, linestyle="--", alpha=0.2)

        # legend only first panel
        if k == 0:
            ax.legend(fontsize=9)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Functional principal component modes of the aligned inner curves. "
            "Each panel shows the mean shape together with plus and minus two "
            "standard deviation deformations along the corresponding mode."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)

def plot_embedded_fpca_variance(beta_aligned, r_aligned, name="fig_synthetic_embedded_fpca_variance"):
    X_aligned = np.array([
        embed_curve(beta_aligned[i], r_aligned[i])
        for i in range(len(beta_aligned))
    ])

    _, _, _, Svals = compute_fpca_modes(X_aligned)

    var = Svals ** 2
    ratio = var / var.sum()
    cumulative = np.cumsum(ratio)

    n_show = min(10, len(ratio))

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.bar(np.arange(1, n_show + 1), ratio[:n_show], label="Explained variance")
    ax.plot(
        np.arange(1, n_show + 1),
        cumulative[:n_show],
        marker="o",
        lw=2,
        label="Cumulative variance"
    )

    ax.set_xlabel("FPCA component")
    ax.set_ylabel("Variance ratio")
    ax.set_title("Embedded FPCA explained variance")
    ax.legend()

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Explained variance ratios from FPCA of the aligned embedded "
            "curve-thickness representation."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)

    var_table = pd.DataFrame({
        "component": np.arange(1, len(ratio) + 1),
        "explained_variance_ratio": ratio,
        "cumulative_variance_ratio": cumulative
    })

    save_latex_table(
        var_table.head(10),
        "synthetic_embedded_fpca_explained_variance",
        "Explained variance ratios for embedded FPCA components.",
        "tab:synthetic_embedded_fpca_explained_variance"
    )


def plot_srvf_tangent_fpca_modes(
    srvf_fpca_payload,
    X_template,
    beta_aligned,
    r_aligned,
    name="fig_synthetic_srvf_tangent_fpca_modes"
):
    """
    Tangent-space FPCA modes at the SRVF Karcher mean.

    Modes are computed in the tangent space at q_mean_lambda,
    mapped back by the exponential map, and reconstructed by inverse SRVF.
    """

    mu = srvf_fpca_payload["mu"]
    pcs = srvf_fpca_payload["pcs"]
    scores = srvf_fpca_payload["scores"]
    n_modes = srvf_fpca_payload["n_modes"]

    beta_mean = enforce_curve_closure(X_template[:, :2])
    r_mean = enforce_scalar_periodicity(X_template[:, 2] / alpha)
    

    # Use same initial point as reconstructed template
    initial_point = X_template[0].copy()

    fig, axes = plt.subplots(n_modes, 2, figsize=(11, 4.2 * n_modes))

    if n_modes == 1:
        axes = np.array([axes])

    for k in range(n_modes):
        pc = pcs[k]
        sd_score = np.std(scores[:, k])
        tau = 2.0 * sd_score

        X_minus, beta_minus, r_minus = reconstruct_srvf_fpca_mode(
            mu=mu,
            pc_flat=pc,
            tau=-tau,
            initial_point=initial_point
        )

        X_plus, beta_plus, r_plus = reconstruct_srvf_fpca_mode(
            mu=mu,
            pc_flat=pc,
            tau=tau,
            initial_point=initial_point
        )

        # Optional: align mode curves roughly to template scale/location
        for X_mode in [X_minus, X_plus]:
            mean_xy_length_data = curve_length_xy(X_template)
            mean_xy_length_mode = curve_length_xy(X_mode)

            xy_scale = mean_xy_length_data / (mean_xy_length_mode + 1e-8)

            xy_cent = np.mean(X_mode[:, :2], axis=0, keepdims=True)
            target_cent = np.mean(X_template[:, :2], axis=0, keepdims=True)

            X_mode[:, :2] = xy_cent + xy_scale * (X_mode[:, :2] - xy_cent)
            X_mode[:, :2] += target_cent - np.mean(X_mode[:, :2], axis=0, keepdims=True)

        beta_minus = enforce_curve_closure(X_minus[:, :2])
        beta_plus = enforce_curve_closure(X_plus[:, :2])

        r_minus = enforce_scalar_periodicity(X_minus[:, 2] / alpha)
        r_plus = enforce_scalar_periodicity(X_plus[:, 2] / alpha)

        # ----------------------------
        # Shape panel
        # ----------------------------
        ax = axes[k, 0]

        # --- raw aligned curves (background) ---
        for beta in beta_aligned:
            ax.plot(
                *close_for_plot(beta).T,
                color="gray",
                alpha=0.15,
                lw=1
            )

        ax.plot(
            *close_for_plot(beta_mean).T,
            color="black",
            lw=3,
            label="Mean"
        )

        ax.plot(
            *close_for_plot(beta_minus).T,
            color="#1f77b4",
            lw=2.5,
            linestyle="--",
            label="-2 SD"
        )

        ax.plot(
            *close_for_plot(beta_plus).T,
            color="#d95f02",
            lw=2.5,
            linestyle="--",
            label="+2 SD"
        )

        ax.set_title(f"SRVF tangent FPCA mode {k + 1}: shape")
        ax.set_aspect("equal")
        ax.grid(True, linestyle="--", alpha=0.2)

        if k == 0:
            ax.legend(fontsize=9)

        # ----------------------------
        # Thickness panel
        # ----------------------------
        ax = axes[k, 1]

        # --- SRVF reconstructed thickness (background) ---
        for r in r_aligned:
            ax.plot(
                s,
                r,
                color="gray",
                alpha=0.15,
                lw=1
            )

        # SRVF Karcher mean
        ax.plot(
            s,
            r_mean,
            color="black",
            lw=3,
            linestyle="-",
            label="SRVF mean"
        )

        ax.plot(
            s,
            r_minus,
            color="#1f77b4",
            lw=2.5,
            linestyle="--",
            label="-2 SD"
        )

        ax.plot(
            s,
            r_plus,
            color="#d95f02",
            lw=2.5,
            linestyle="--",
            label="+2 SD"
        )

        ax.set_title(f"SRVF tangent FPCA mode {k + 1}: thickness")
        ax.set_xlabel(r"$s$")
        ax.set_ylabel(r"$r_{\mathrm{SRVF}}(s)$")
        ax.grid(True, linestyle="--", alpha=0.2)

        if k == 0:
                ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Primary FPCA analysis on the SRVF Hilbert sphere. "
            "Principal directions are computed in the tangent space at the SRVF "
            "Karcher mean, mapped back to the sphere using the exponential map, "
            "and reconstructed by inverse SRVF to show coupled shape and thickness variation."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)


    
def plot_closure_verification(
    beta_mean,
    r_mean,
    q_raw=None,
    q_hat=None,
    q_warped_stage=None,
    q_aligned_stage=None,
    X_warped=None,
    X_final=None,
    q_mean=None,
    name="fig_closure_verification"
):
    """
    Closure verification figure.

    Left:
        Mean closed inner/outer representation.

    Right-top:
        Geometric endpoint closure:
            ||X(1) - X(0)||

    Right-bottom:
        SRVF integral closure condition:
            || integral q(s)||q(s)|| ds ||
    """

    beta_mean = enforce_curve_closure(beta_mean)
    r_mean = enforce_scalar_periodicity(r_mean)

    beta_out_mean = reconstruct_outer(beta_mean, r_mean)

    fig = plt.figure(figsize=(14, 6))

    gs = fig.add_gridspec(
        2, 2,
        width_ratios=[1.05, 1.25],
        height_ratios=[1, 1],
        wspace=0.28,
        hspace=0.35
    )

    ax_left = fig.add_subplot(gs[:, 0])
    ax_geo = fig.add_subplot(gs[0, 1])
    ax_srvf = fig.add_subplot(gs[1, 1])

    # ========================================================
    # Left panel: mean closed representation
    # ========================================================
    B_in = close_for_plot(beta_mean)
    B_out = close_for_plot(beta_out_mean)

    poly_x = np.concatenate([B_out[:, 0], B_in[::-1, 0]])
    poly_y = np.concatenate([B_out[:, 1], B_in[::-1, 1]])

    ax_left.fill(
        poly_x,
        poly_y,
        alpha=0.30,
        label="Mean thickness band"
    )

    ax_left.plot(
        B_in[:, 0],
        B_in[:, 1],
        lw=3,
        label="Mean inner curve"
    )

    ax_left.plot(
        B_out[:, 0],
        B_out[:, 1],
        "--",
        lw=3,
        label="Mean outer curve"
    )

    ax_left.scatter(
        B_in[0, 0],
        B_in[0, 1],
        s=60,
        zorder=5,
        label="Start/end point"
    )

    ax_left.set_title("Mean closed representation")
    ax_left.axis("equal")
    ax_left.grid(True, linestyle="--", alpha=0.20)
    ax_left.legend(fontsize=9)

    # ========================================================
    # Right-top panel: geometric endpoint closure
    # ========================================================
    geometric_series = {}

    if X_warped is not None:
        geometric_series["after warping"] = np.array([
            endpoint_closure_error(X_warped[i])
            for i in range(len(X_warped))
        ])

    if X_final is not None:
        geometric_series["after rotation"] = np.array([
            endpoint_closure_error(X_final[i])
            for i in range(len(X_final))
        ])

    for label, vals in geometric_series.items():
        ax_geo.plot(
            np.arange(len(vals)),
            vals,
            marker="o",
            lw=2,
            label=label
        )

    ax_geo.axhline(
        0.0,
        color="black",
        linestyle="--",
        lw=2,
        label="target = 0"
    )

    ax_geo.set_title("Geometric endpoint closure")
    ax_geo.set_xlabel("curve index")
    ax_geo.set_ylabel(r"$\|X(1)-X(0)\|$")
    ax_geo.grid(True, linestyle="--", alpha=0.30)
    ax_geo.legend(fontsize=8)

    # ========================================================
    # Right-bottom panel: SRVF integral closure condition
    # ========================================================
    srvf_series = {}

    if q_raw is not None:
        srvf_series["raw SRVF"] = np.array([
            np.linalg.norm(closure_vector_from_srvf(q_raw[i]))
            for i in range(len(q_raw))
        ])

    if q_hat is not None:
        srvf_series["normalized SRVF"] = np.array([
            np.linalg.norm(closure_vector_from_srvf(q_hat[i]))
            for i in range(len(q_hat))
        ])

    if q_warped_stage is not None:
        srvf_series["after warping"] = np.array([
            np.linalg.norm(closure_vector_from_srvf(q_warped_stage[i]))
            for i in range(len(q_warped_stage))
        ])

    if q_aligned_stage is not None:
        srvf_series["after rotation"] = np.array([
            np.linalg.norm(closure_vector_from_srvf(q_aligned_stage[i]))
            for i in range(len(q_aligned_stage))
        ])

    eps_plot = 1e-16

    line_styles = {
        "raw SRVF": "-",
        "normalized SRVF": "--",
        "after warping": "-.",
        "after rotation": ":"
    }

    markers = {
        "raw SRVF": "o",
        "normalized SRVF": "s",
        "after warping": "^",
        "after rotation": "D"
    }

    for label, vals in srvf_series.items():
        vals_plot = np.maximum(vals, eps_plot)

        ax_srvf.plot(
            np.arange(len(vals_plot)),
            vals_plot,
            linestyle=line_styles.get(label, "-"),
            marker=markers.get(label, "o"),
            markersize=4,
            lw=2,
            alpha=0.85,
            label=label
        )

    

    if q_mean is not None:
        mean_val = max(np.linalg.norm(closure_vector_from_srvf(q_mean)), eps_plot)
        ax_srvf.axhline(
            mean_val,
            color="black",
            linestyle="--",
            lw=2,
            label="mean SRVF"
        )
    else:
        ax_srvf.axhline(
            0.0,
            color="black",
            linestyle="--",
            lw=2,
            label="target = 0"
        )

    ax_srvf.set_title("SRVF integral closure condition")
    ax_srvf.set_yscale("log")
    ax_srvf.set_xlabel("curve index")
    ax_srvf.set_ylabel(r"$\left\|\int q(s)\|q(s)\|\,ds\right\|$")
    ax_srvf.grid(True, linestyle="--", alpha=0.30)
    ax_srvf.legend(fontsize=8)

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Closure verification for the embedded closed-curve representation. "
            "The left panel shows the reconstructed mean inner and outer curves. "
            "The upper-right panel reports geometric endpoint closure, while the "
            "lower-right panel reports the SRVF integral closure condition computed "
            "directly from stored SRVF stages."
        ),
        label="fig:closure_verification"
    )

    plt.close(fig)

def align_mode_to_mean(beta_mode, beta_mean, do_rotation=False):
    """
    Display-only alignment of FPCA mode curve to its mean.
    Removes centroid drift; optional SO(2) visual alignment.
    """
    beta_mode = enforce_curve_closure(beta_mode.copy())
    beta_mean = enforce_curve_closure(beta_mean.copy())

    cent_mode = np.mean(beta_mode, axis=0, keepdims=True)
    cent_mean = np.mean(beta_mean, axis=0, keepdims=True)

    beta_mode = beta_mode - cent_mode + cent_mean

    if do_rotation:
        R_vis = optimal_rotation_2d(beta_mode, beta_mean)
        beta_mode = beta_mode @ R_vis
        beta_mode = beta_mode - np.mean(beta_mode, axis=0, keepdims=True) + cent_mean

    return enforce_curve_closure(beta_mode)

def plot_shape_fpca_comparison(
    beta_aligned,
    srvf_beta_reconstructed,
    srvf_fpca_payload,
    X_template,
    name="fig_shape_fpca_comparison"
):
    """
    Compare classical aligned-domain shape FPCA with SRVF tangent-space shape FPCA.

    Rows    : modes 1, 2, 3
    Column 1: Classical FPCA of beta
    Column 2: SRVF tangent FPCA of beta
    """

    # Classical FPCA on beta_aligned
    beta_mean_flat, beta_pcs, beta_scores, _ = compute_fpca_modes(beta_aligned)
    beta_mean_classical = beta_mean_flat.reshape(N, 2)

    # SRVF tangent FPCA
    mu = srvf_fpca_payload["mu"]
    pcs_srvf = srvf_fpca_payload["pcs"]
    scores_srvf = srvf_fpca_payload["scores"]
    n_modes = min(3, srvf_fpca_payload["n_modes"])
    joint = srvf_fpca_payload.get('joint_display')
    if joint is None:
        # Backward-compatible direct calls use the supplied display reference.
        q_all = [exp_map_sphere(v.reshape(mu.shape), mu)
                 for v in srvf_fpca_payload['tangent_vectors']]
        joint = _srvf_numerics.joint_mode_curves(
            mu, q_all, [X_template], compute_srvf_tangent_fpca,
            exp_map_sphere, alpha, n_modes=n_modes, endpoint_duplicated=True
        )

    beta_mean_srvf = joint['mean'][:, :2]
    initial_point = X_template[0].copy()

    fig, axes = plt.subplots(n_modes, 2, figsize=(10, 4.2 * n_modes))

    if n_modes == 1:
        axes = np.array([axes])

    lim = 1.6

    for k in range(n_modes):
        # -----------------------------
        # Classical FPCA mode
        # -----------------------------
        pc = beta_pcs[k].reshape(N, 2)
        sd_score = np.std(beta_scores[:, k])

        beta_minus = beta_mean_classical - 2.0 * sd_score * pc
        beta_plus = beta_mean_classical + 2.0 * sd_score * pc

        beta_mean_classical = enforce_curve_closure(beta_mean_classical)
        beta_minus = align_mode_to_mean(beta_minus, beta_mean_classical, do_rotation=False)
        beta_plus = align_mode_to_mean(beta_plus, beta_mean_classical, do_rotation=False)

        ax = axes[k, 0]

        for beta in beta_aligned:
            B = close_for_plot(beta)
            ax.plot(B[:, 0], B[:, 1], color="gray", alpha=0.15, lw=0.8)

        ax.plot(*close_for_plot(beta_mean_classical).T,
                color="black", lw=3, label="Mean")
        ax.plot(*close_for_plot(beta_minus).T,
                color="#1f77b4", lw=2.5, linestyle="--", label="-2 SD")
        ax.plot(*close_for_plot(beta_plus).T,
                color="#d95f02", lw=2.5, linestyle="--", label="+2 SD")

        ax.set_title(f"Mode {k + 1}: Classical FPCA")
        ax.set_xlabel(r"$\beta_x(s)$")
        ax.set_ylabel(r"$\beta_y(s)$")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")
        ax.grid(True, linestyle="--", alpha=0.2)

        if k == 0:
            ax.legend(fontsize=8, loc="upper right")

        # -----------------------------
        # SRVF tangent FPCA mode
        # -----------------------------
        X_minus, X_plus = joint['modes'][k]
        beta_minus_srvf, beta_plus_srvf = X_minus[:, :2], X_plus[:, :2]

        ax = axes[k, 1]

        for beta in srvf_beta_reconstructed:
            B = close_for_plot(beta)
            ax.plot(B[:, 0], B[:, 1], color="gray", alpha=0.15, lw=0.8)

        ax.plot(*close_for_plot(beta_mean_srvf).T,
                color="black", lw=3, label="SRVF mean")
        ax.plot(*close_for_plot(beta_minus_srvf).T,
                color="#1f77b4", lw=2.5, linestyle="--", label="-2 SD")
        ax.plot(*close_for_plot(beta_plus_srvf).T,
                color="#d95f02", lw=2.5, linestyle="--", label="+2 SD")

        ax.set_title(f"Mode {k + 1}: SRVF tangent FPCA")
        ax.set_xlabel(r"$\beta_x(s)$")
        ax.set_ylabel(r"$\beta_y(s)$")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")
        ax.grid(True, linestyle="--", alpha=0.2)

        if k == 0:
            ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Comparison of classical aligned-domain shape FPCA and SRVF tangent-space "
            "shape FPCA. Rows correspond to the first three modes. The left column "
            "shows classical FPCA of aligned planar curves, while the right column "
            "shows tangent-space FPCA modes mapped back from the SRVF Karcher mean."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)


def plot_thickness_fpca_comparison(
    r_aligned,
    srvf_r_reconstructed,
    srvf_fpca_payload,
    X_template,
    name="fig_thickness_fpca_comparison"
):
    """
    Compare classical aligned-domain thickness FPCA with SRVF tangent-space
    thickness modes.

    Rows:
        Mode 1, Mode 2, Mode 3

    Columns:
        Left  : Classical FPCA of aligned thickness r(s)
        Right : SRVF tangent FPCA induced thickness r_SRVF(s)
    """

    # --------------------------------------------------------
    # Classical FPCA on aligned thickness
    # --------------------------------------------------------
    r_mean_classical, r_pcs, r_scores, _ = compute_fpca_modes(r_aligned)

    # --------------------------------------------------------
    # SRVF tangent FPCA information
    # --------------------------------------------------------
    mu = srvf_fpca_payload["mu"]
    pcs_srvf = srvf_fpca_payload["pcs"]
    scores_srvf = srvf_fpca_payload["scores"]
    n_modes = min(3, srvf_fpca_payload["n_modes"])
    joint = srvf_fpca_payload.get('joint_display')
    if joint is None:
        # Backward-compatible direct calls use the supplied display reference.
        q_all = [exp_map_sphere(v.reshape(mu.shape), mu)
                 for v in srvf_fpca_payload['tangent_vectors']]
        joint = _srvf_numerics.joint_mode_curves(
            mu, q_all, [X_template], compute_srvf_tangent_fpca,
            exp_map_sphere, alpha, n_modes=n_modes, endpoint_duplicated=True
        )

    # SRVF Karcher mean thickness
    r_mean_srvf = joint['mean'][:, 2]/alpha

    initial_point = X_template[0].copy()

    fig, axes = plt.subplots(
        n_modes,
        2,
        figsize=(12, 3.8 * n_modes),
        sharex=True
    )

    if n_modes == 1:
        axes = np.array([axes])


    # Separate y-axis limits for classical and SRVF panels
    ymin_classical = np.min(r_aligned)
    ymax_classical = np.max(r_aligned)
    ypad_classical = 0.08 * (ymax_classical - ymin_classical)

    ymin_srvf = min(
        np.min(srvf_r_reconstructed),
        np.min(r_mean_srvf)
    )
    ymax_srvf = max(
        np.max(srvf_r_reconstructed),
        np.max(r_mean_srvf)
    )
    ypad_srvf = 0.08 * (ymax_srvf - ymin_srvf)

   

    for k in range(n_modes):

        # ====================================================
        # Left column: Classical thickness FPCA
        # ====================================================
        pc = r_pcs[k]
        sd_score = np.std(r_scores[:, k])

        r_minus = r_mean_classical - 2.0 * sd_score * pc
        r_plus = r_mean_classical + 2.0 * sd_score * pc

        ax = axes[k, 0]

        for r in r_aligned:
            ax.plot(
                s,
                r,
                color="gray",
                alpha=0.15,
                lw=0.8
            )

        ax.plot(
            s,
            r_mean_classical,
            color="black",
            lw=3,
            label="Mean"
        )

        ax.plot(
            s,
            r_minus,
            color="#1f77b4",
            lw=2.5,
            linestyle="--",
            label="-2 SD"
        )

        ax.plot(
            s,
            r_plus,
            color="#d95f02",
            lw=2.5,
            linestyle="--",
            label="+2 SD"
        )

        ax.set_title(f"Mode {k + 1}: Classical FPCA")
        ax.set_ylabel(r"$r(s)$")
        ax.set_ylim(ymin_classical - ypad_classical, ymax_classical + ypad_classical)
        ax.grid(True, linestyle="--", alpha=0.2)

        if k == n_modes - 1:
            ax.set_xlabel(r"$s$")

        if k == 0:
            ax.legend(fontsize=8, loc="upper right")

        # ====================================================
        # Right column: SRVF tangent FPCA thickness
        # ====================================================
        X_minus, X_plus = joint['modes'][k]
        r_minus_srvf, r_plus_srvf = X_minus[:, 2]/alpha, X_plus[:, 2]/alpha

        # Update y-limits to include reconstructed SRVF mode curves
        local_ymin = min(
            ymin_srvf,
            np.min(r_minus_srvf),
            np.min(r_plus_srvf)
        )
        local_ymax = max(
            ymax_srvf,
            np.max(r_minus_srvf),
            np.max(r_plus_srvf)
        )
        local_ypad = 0.08 * (local_ymax - local_ymin)

        ax = axes[k, 1]

        for r in srvf_r_reconstructed:
            ax.plot(
                s,
                r,
                color="gray",
                alpha=0.15,
                lw=0.8
            )

        ax.plot(
            s,
            r_mean_srvf,
            color="black",
            lw=3,
            linestyle="-",
            label="Mean"
        )

        ax.plot(
            s,
            r_minus_srvf,
            color="#1f77b4",
            lw=2.5,
            linestyle="--",
            label="-2 SD"
        )

        ax.plot(
            s,
            r_plus_srvf,
            color="#d95f02",
            lw=2.5,
            linestyle="--",
            label="+2 SD"
        )

        ax.set_title(f"Mode {k + 1}: SRVF tangent FPCA")
        ax.set_ylabel(r"$r_{\mathrm{SRVF}}(s)$")
        ax.set_ylim(local_ymin - local_ypad, local_ymax + local_ypad)
        ax.grid(True, linestyle="--", alpha=0.2)

        if k == n_modes - 1:
            ax.set_xlabel(r"$s$")

        if k == 0:
            ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()

    save_figure(
        fig,
        name,
        caption=(
            "Comparison of classical aligned-domain thickness FPCA and SRVF "
            "tangent-space FPCA. Rows correspond to the first three modes. "
            "The left column shows FPCA modes of aligned thickness functions "
            r"$r(s)$, while the right column shows thickness components induced "
            "by SRVF tangent-space perturbations reconstructed from the Karcher mean."
        ),
        label=f"fig:{name}"
    )

    plt.close(fig)
    
# ============================================================
# 10. One simulation
# ============================================================

def run_one_simulation(sim_id):
    """
    Run one complete synthetic simulation repetition.

    Pipeline:
        1. Generate closed synthetic curve-thickness data.
        2. Select lambda by K-fold CV using penalized DP alignment.
        3. Align the full dataset using selected lambda.
        4. Optionally refine the aligned representatives using fdasrsf.
        5. Compute the final Karcher mean for lambda_star.
        6. Evaluate alignment performance and closure diagnostics.
        7. Return results and representative objects for figures.
    """

    global rng
    rng = np.random.default_rng(SEED + sim_id)

    print("==========================================")
    print(f"Simulation {sim_id + 1}/{N_SIM}")
    print("==========================================")

    # --------------------------------------------------------
    # 1. Generate synthetic closed curve-thickness data
    # --------------------------------------------------------
    beta_raw, r_raw, beta_out_raw, X_raw, gamma_true, latent = generate_dataset(
        sim_id
    )

    q_raw, q_hat = compute_all_srvfs(X_raw)

    # --------------------------------------------------------
    # 2. Select lambda by K-fold CV
    # --------------------------------------------------------
    best_lambda, lambda_table, lambda_summary = select_lambda_5fold_cv(
        beta_raw=beta_raw,
        r_raw=r_raw,
        X_raw=X_raw,
        sim_id=sim_id
    )

    # --------------------------------------------------------
    # 3. Final DP alignment using selected lambda
    # --------------------------------------------------------
    final_alignment = run_alignment(
        beta_raw=beta_raw,
        r_raw=r_raw,
        X_raw=X_raw,
        lambda_warp=best_lambda
    )

    (
        beta_warped,
        r_warped,
        X_warped,
        beta_aligned_dp,
        r_aligned_dp,
        X_aligned_dp,
        gammas_dp,
        q_warped_stage,
        q_aligned_stage,
        q_mean_lambda,
        X_template_from_alignment
    ) = final_alignment

    # Default final objects are DP outputs
    X_final = np.array([
        enforce_curve_closure(X_aligned_dp[i])
        for i in range(len(X_aligned_dp))
    ])

    gammas_final = gammas_dp.copy()
    refinement_used = False
    mean_curve_fdasrsf = None


    # --------------------------------------------------------
    # 5. Recover beta and r from final embedded curves
    # --------------------------------------------------------
    beta_final = X_final[:, :, :2]
    r_final = X_final[:, :, 2] / alpha

    beta_final = np.array([
        enforce_curve_closure(beta_final[i])
        for i in range(len(beta_final))
    ])

    r_final = np.array([
        enforce_scalar_periodicity(r_final[i])
        for i in range(len(r_final))
    ])

    X_final = np.array([
        embed_curve(beta_final[i], r_final[i])
        for i in range(len(beta_final))
    ])

    # --------------------------------------------------------
    # 6. Final Karcher summary for selected lambda
    # --------------------------------------------------------
    q_final_aligned = np.array([
        normalize_srvf(q_aligned_stage[i])
        for i in range(len(q_aligned_stage))
    ])

    final_distances = np.array([
        sphere_geodesic_distance(q_final_aligned[i], q_mean_lambda)
        for i in range(len(q_final_aligned))
    ])

    final_penalties = np.array([
        warping_penalty_integral(gammas_final[i])
        for i in range(len(gammas_final))
    ])

    karcher_summary = {
        "lambda_star": best_lambda,
        "mean_srvf_geodesic_distance": final_distances.mean(),
        "sd_srvf_geodesic_distance": final_distances.std(),
        "mean_warping_penalty": final_penalties.mean(),
        "sd_warping_penalty": final_penalties.std()
    }

    save_json({
        'geometry': 'ambient_sphere_approximation',
        'mean': {k: v.tolist() if isinstance(v, np.ndarray) else v
                 for k, v in _srvf_numerics.closure_diagnostics(q_mean_lambda, True).items()},
        'aligned_closure_norms': [_srvf_numerics.closure_diagnostics(q, True)['closure_norm']
                                 for q in q_final_aligned]
    }, f'closure_diagnostics_sim_{sim_id:03d}')
    print("Final Karcher mean reused from final alignment for lambda* =", best_lambda)
    print(
        "Mean SRVF geodesic distance =",
        karcher_summary["mean_srvf_geodesic_distance"]
    )
    print(
        "Mean warping penalty =",
        karcher_summary["mean_warping_penalty"]
    )

    
    # --------------------------------------------------------
    # 6b. Tangent-space FPCA at SRVF Karcher mean
    # --------------------------------------------------------
    srvf_fpca_payload = compute_srvf_tangent_fpca(
        q_aligned=q_final_aligned,
        mu=q_mean_lambda,
        n_modes=3
    )

    # --------------------------------------------------------
    srvf_fpca_payload['joint_display'] = _srvf_numerics.joint_mode_curves(
        q_mean_lambda, q_final_aligned, X_final, compute_srvf_tangent_fpca,
        exp_map_sphere, alpha, n_modes=3, endpoint_duplicated=True
    )
    save_json({'geometry': 'ambient_sphere_approximation', 'diagnostics': [
        {k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in row.items()}
        for row in srvf_fpca_payload['joint_display']['diagnostics']
    ]}, f'joint_modes_closure_sim_{sim_id:03d}')
    # Reconstruct SRVF-domain curves for plotting
    # --------------------------------------------------------
    X_srvf_rec, beta_srvf_rec, r_srvf_rec = reconstruct_curves_from_srvfs(
        q_aligned=q_final_aligned,
        X_final=X_final
    )

    # SRVF tangent FPCA evaluated in its native SRVF geometry
    srvf_mean_error = srvf_mean_only_error(
        q_aligned=q_final_aligned,
        mu=q_mean_lambda
    )

    srvf_fpca_error_table = compare_srvf_tangent_fpca_reconstruction(
        q_aligned=q_final_aligned,
        srvf_fpca_payload=srvf_fpca_payload,
        n_modes_list=[1, 2, 3]
    )

    srvf_fpca_error_table["mean_only_srvf_geodesic_error"] = srvf_mean_error

    srvf_fpca_error_table["improvement_over_mean_percent"] = (
        100.0
        * (
            srvf_mean_error
            - srvf_fpca_error_table["srvf_tangent_fpca_geodesic_error"]
        )
        / srvf_mean_error
    )

    srvf_fpca_error_table.insert(0, "sim", sim_id)

    # Optional supplementary comparison in curve/thickness domain
    fpca_comparison_table = compare_aligned_fpca_vs_srvf_tangent_fpca(
        beta_aligned=beta_final,
        r_aligned=r_final,
        X_final=X_final,
        srvf_fpca_payload=srvf_fpca_payload,
        n_modes_list=[1, 2, 3]
    )

    fpca_comparison_table.insert(0, "sim", sim_id)



    # --------------------------------------------------------
    # 7. Template for curve-space evaluation and plotting
    # --------------------------------------------------------
    X_template = enforce_curve_closure(invert_srvf_from_q(q_mean_lambda))
    mean_xy_length = np.mean([curve_length_xy(curve) for curve in X_final])
    target_center = np.mean(X_final, axis=(0, 1))
    X_template = restore_embedded_scale(X_template, mean_xy_length, target_center)
    X_template = enforce_curve_closure(X_template)

    # --------------------------------------------------------
    # 8. Evaluation
    # --------------------------------------------------------
    eval_table = evaluate_alignment_to_template(
        beta_raw=beta_raw,
        r_raw=r_raw,
        X_raw=X_raw,
        beta_aligned=beta_final,
        r_aligned=r_final,
        X_aligned=X_final,
        X_template=X_template, q_reference=q_mean_lambda, q_aligned=q_final_aligned
    )

    # Pairwise thickness evaluation (true alignment effect)
    before_pw_srvf = mean_pairwise_srvf_geodesic(X_raw, q_all=q_hat)
    after_pw_srvf  = mean_pairwise_srvf_geodesic(X_final, q_all=q_final_aligned)

    before_pw_shape = mean_pairwise_shape(beta_raw)
    after_pw_shape  = mean_pairwise_shape(beta_final)

    before_pw_thick = mean_pairwise_thickness(r_raw)
    after_pw_thick  = mean_pairwise_thickness(r_final)

    pairwise_rows = pd.DataFrame([
        {
            "Metric": "Pairwise SRVF geodesic distance",
            "Before alignment": before_pw_srvf,
            "After alignment": after_pw_srvf,
            "Change": 100 * (before_pw_srvf - after_pw_srvf) / before_pw_srvf
        },
        {
            "Metric": "Pairwise shape distance",
            "Before alignment": before_pw_shape,
            "After alignment": after_pw_shape,
            "Change": 100 * (before_pw_shape - after_pw_shape) / before_pw_shape
        },
        {
            "Metric": "Pairwise thickness distance",
            "Before alignment": before_pw_thick,
            "After alignment": after_pw_thick,
            "Change": 100 * (before_pw_thick - after_pw_thick) / before_pw_thick
        }
    ])
    

    eval_table = pd.concat([eval_table, pairwise_rows], ignore_index=True)

    eval_table.insert(0, "sim", sim_id)
    eval_table["best_lambda"] = best_lambda
    eval_table["fdasrsf_refinement"] = refinement_used
    eval_table["gamma_deviation_mse"] = gamma_deviation_mse(gammas_final)

    eval_table["karcher_mean_srvf_distance"] = (
        karcher_summary["mean_srvf_geodesic_distance"]
    )
    eval_table["karcher_mean_warping_penalty"] = (
        karcher_summary["mean_warping_penalty"]
    )

    # --------------------------------------------------------
    # 9. Closure diagnostics
    # --------------------------------------------------------
    closure_raw = np.mean([
        endpoint_closure_error(X_raw[i])
        for i in range(len(X_raw))
    ])

    closure_final = np.mean([
        endpoint_closure_error(X_final[i])
        for i in range(len(X_final))
    ])

    srvf_closure_final = np.mean([
        np.linalg.norm(closure_vector_from_srvf(srvf(X_final[i])))
        for i in range(len(X_final))
    ])

    eval_table["endpoint_closure_raw"] = closure_raw
    eval_table["endpoint_closure_final"] = closure_final
    eval_table["srvf_closure_final"] = srvf_closure_final

    # --------------------------------------------------------
    # 10. Lambda tables
    # --------------------------------------------------------
    lambda_table["sim"] = sim_id
    lambda_table["best_lambda"] = best_lambda
    lambda_table["fdasrsf_refinement"] = refinement_used

    lambda_summary["sim"] = sim_id
    lambda_summary["best_lambda"] = best_lambda
    lambda_summary["fdasrsf_refinement"] = refinement_used

    lambda_summary["final_mean_srvf_geodesic_distance"] = (
        karcher_summary["mean_srvf_geodesic_distance"]
    )
    lambda_summary["final_mean_warping_penalty"] = (
        karcher_summary["mean_warping_penalty"]
    )

    # --------------------------------------------------------
    # 11. Representative payload for figures
    # --------------------------------------------------------
    pipeline_payload = None

    if sim_id == 0:
        q_final_raw, q_final_hat = compute_all_srvfs(X_final)

        pipeline_payload = {
            "beta_raw": beta_raw,
            "beta_out_raw": beta_out_raw,
            "q_raw": q_raw,
            "q_hat": q_hat,

            "beta_warped": beta_warped,
            "r_warped": r_warped,
            "X_warped": X_warped,
            "q_warped_stage": q_warped_stage,
            "q_aligned_stage": q_aligned_stage,

            "beta_aligned": beta_final,
            "r_aligned": r_final,
            "srvf_beta_reconstructed": beta_srvf_rec,
            "srvf_r_reconstructed": r_srvf_rec,
            "X_final": X_final,

            "gammas": gammas_final,
            "X_template": X_template,

            "q_mean_lambda": q_mean_lambda,
            "q_final_aligned": q_final_aligned,
            "srvf_fpca_payload": srvf_fpca_payload,
            "q_final_raw": q_final_raw,
            "q_final_hat": q_final_hat,

            "final_distances": final_distances,
            "final_penalties": final_penalties,
            "karcher_summary": karcher_summary,

            "best_lambda": best_lambda,
            "fdasrsf_refinement": refinement_used
        }

    # --------------------------------------------------------
    # 12. Return all outputs
    # --------------------------------------------------------
    return {
        "evaluation": eval_table,
        "lambda": lambda_table,
        "lambda_summary": lambda_summary,
        "fpca_comparison": fpca_comparison_table,
        "srvf_fpca_error": srvf_fpca_error_table,
        "pipeline_payload": pipeline_payload
    }

# ============================================================
# 11. Main
# ============================================================

def main():
    print("==========================================")
    print("REPEATED SYNTHETIC OFFSET-SRVF PIPELINE")
    print("K-FOLD CV + SRVF KARCHER MEAN")
    print("==========================================")
    print(f"fdasrsf available: {HAS_FDASRSF}")

    all_eval = []
    all_lambda = []
    all_lambda_summary = []
    all_fpca_comparison = []
    all_srvf_fpca_error = []

    pipeline_payload = None

    for sim_id in range(N_SIM):
        out = run_one_simulation(sim_id)

        all_eval.append(out["evaluation"])
        all_lambda.append(out["lambda"])
        all_lambda_summary.append(out["lambda_summary"])
        all_fpca_comparison.append(out["fpca_comparison"])
        all_srvf_fpca_error.append(out["srvf_fpca_error"])

        if sim_id == 0:
            pipeline_payload = out["pipeline_payload"]


    eval_results = pd.concat(all_eval, ignore_index=True)
    lambda_results = pd.concat(all_lambda, ignore_index=True)
    lambda_summary_results = pd.concat(all_lambda_summary, ignore_index=True)
    fpca_comparison_results = pd.concat(all_fpca_comparison, ignore_index=True)
    srvf_fpca_error_results = pd.concat(all_srvf_fpca_error, ignore_index=True)


    srvf_fpca_error_results.to_csv(
        TAB_DIR / "synthetic_srvf_tangent_fpca_error_results.csv",
        index=False
    )

    save_latex_table(
        srvf_fpca_error_results,
        "synthetic_srvf_tangent_fpca_error_results",
        "SRVF tangent-space FPCA reconstruction error measured by Hilbert-sphere geodesic distance.",
        "tab:synthetic_srvf_tangent_fpca_error_results"
    )

    eval_results.to_csv(
        TAB_DIR / "synthetic_alignment_repeated_results.csv",
        index=False
    )

    lambda_results.to_csv(
        TAB_DIR / "synthetic_lambda_repeated_results.csv",
        index=False
    )

    lambda_summary_results.to_csv(
        TAB_DIR / "synthetic_lambda_cv_summary_repeated.csv",
        index=False
    )

    fpca_comparison_results.to_csv(
        TAB_DIR / "synthetic_fpca_comparison_results.csv",
        index=False
    )

    save_latex_table(
        fpca_comparison_results,
        "synthetic_fpca_comparison_results",
        "Comparison of aligned-domain FPCA and SRVF tangent-space FPCA reconstruction errors.",
        "tab:synthetic_fpca_comparison_results"
    )

    fpca_comparison_results.to_csv(
        TAB_DIR / "synthetic_fpca_comparison_results.csv",
        index=False
    )


    alignment_summary = summarize_repeated_results(
        eval_results,
        group_cols=["Metric"],
        metric_cols=[
            "Before alignment",
            "After alignment",
            "Change",
            "gamma_deviation_mse"
        ]
    )

    save_latex_table(
        alignment_summary,
        "synthetic_alignment_repeated_summary",
        "Repeated synthetic alignment performance summarized as mean and standard deviation.",
        "tab:synthetic_alignment_repeated_summary"
    )

    lambda_frequency = (
        lambda_summary_results[["sim", "best_lambda"]]
        .drop_duplicates()
        .groupby("best_lambda")
        .size()
        .reset_index(name="frequency")
    )

    save_latex_table(
        lambda_frequency,
        "synthetic_lambda_selection_frequency",
        "Frequency of selected lambda values across repeated synthetic simulations.",
        "tab:synthetic_lambda_selection_frequency"
    )

    plot_lambda_frequency(lambda_summary_results)
    plot_lambda_curve_repeated(lambda_results)

    if pipeline_payload is not None:
        plot_pipeline_example(
            beta_raw=pipeline_payload["beta_raw"],
            beta_out_raw=pipeline_payload["beta_out_raw"],
            q_raw=pipeline_payload["q_raw"],
            q_hat=pipeline_payload["q_hat"],
            beta_warped=pipeline_payload["beta_warped"],
            beta_aligned=pipeline_payload["beta_aligned"],
            r_aligned=pipeline_payload["r_aligned"],
            gammas=pipeline_payload["gammas"],
            X_template=pipeline_payload["X_template"],
            srvf_r_reconstructed=pipeline_payload["srvf_r_reconstructed"]
        )

        plot_closure_verification(
            beta_mean=pipeline_payload["X_template"][:, :2],
            r_mean=pipeline_payload["X_template"][:, 2] / alpha,
            q_raw=pipeline_payload["q_raw"],
            q_hat=pipeline_payload["q_hat"],
            q_warped_stage=pipeline_payload["q_warped_stage"],
            q_aligned_stage=pipeline_payload["q_aligned_stage"],
            X_warped=pipeline_payload["X_warped"],
            X_final=pipeline_payload["X_final"],
            q_mean=pipeline_payload["q_mean_lambda"],
            name="fig_closure_verification"
        )

        plot_thickness_fpca_modes(
            pipeline_payload["r_aligned"],
            name="fig_synthetic_thickness_fpca_modes"
        )

        plot_shape_fpca_modes(
            pipeline_payload["beta_aligned"],
            pipeline_payload["r_aligned"],
            name="fig_synthetic_shape_fpca_modes"
        )

        plot_embedded_fpca_variance(
            pipeline_payload["beta_aligned"],
            pipeline_payload["r_aligned"],
            name="fig_synthetic_embedded_fpca_variance"
        )

        plot_srvf_tangent_fpca_modes(
            srvf_fpca_payload=pipeline_payload["srvf_fpca_payload"],
            X_template=pipeline_payload["X_template"],
            beta_aligned=pipeline_payload["srvf_beta_reconstructed"],
            r_aligned=pipeline_payload["srvf_r_reconstructed"],
            name="fig_synthetic_srvf_tangent_fpca_modes"
        )


        plot_shape_fpca_comparison(
            beta_aligned=pipeline_payload["beta_aligned"],
            srvf_beta_reconstructed=pipeline_payload["srvf_beta_reconstructed"],
            srvf_fpca_payload=pipeline_payload["srvf_fpca_payload"],
            X_template=pipeline_payload["X_template"],
            name="fig_shape_fpca_comparison"
        )

        

        plot_thickness_fpca_comparison(
            r_aligned=pipeline_payload["r_aligned"],
            srvf_r_reconstructed=pipeline_payload["srvf_r_reconstructed"],
            srvf_fpca_payload=pipeline_payload["srvf_fpca_payload"],
            X_template=pipeline_payload["X_template"],
            name="fig_thickness_fpca_comparison"
        )

    run_info = {
        "K": K,
        "N": N,
        "N_SIM": N_SIM,
        "N_FOLDS": N_FOLDS,
        "alpha": alpha,
        "PHASE_MODE": PHASE_MODE,
        "AMPLITUDE_MODE": AMPLITUDE_MODE,
        "RANDOM_STATE": RANDOM_STATE,
        "LAMBDA_GRID": LAMBDA_GRID,
        "N_ITER_ALIGN": N_ITER_ALIGN,
        "USE_CYCLIC_PREALIGNMENT": USE_CYCLIC_PREALIGNMENT,
        "fdasrsf_available": HAS_FDASRSF
    }

    save_json(run_info, "synthetic_run_info")

    with open(OUT_DIR / "README_outputs.txt", "w") as f:
        f.write(
            "Repeated Synthetic Offset-SRVF Pipeline Outputs\n"
            "==============================================\n\n"
            "Each simulation generates a new synthetic curve-thickness dataset.\n"
            "The warping parameter lambda is selected using five-fold cross-validation.\n"
            "The CV criterion is the squared SRVF geodesic validation error to the\n"
            "Hilbert-sphere Karcher mean of aligned training SRVFs.\n\n"
            "After lambda selection, the full synthetic dataset is aligned using the\n"
            "selected lambda. Alignment performance is summarized across repetitions.\n\n"
            f"N_SIM: {N_SIM}\n"
            f"N_FOLDS: {N_FOLDS}\n"
            f"K: {K}\n"
            f"N: {N}\n"
            f"alpha: {alpha}\n"
            f"fdasrsf available: {HAS_FDASRSF}\n\n"
            "Folders:\n"
            "  figures/ : PNG, PDF, EPS figures\n"
            "  tables/  : CSV and LaTeX tables\n"
            "  data/    : JSON run information\n"
            "  tex/     : LaTeX figure snippets\n"
        )

    print("==========================================")
    print("DONE")
    print("Outputs saved in:", OUT_DIR)
    print("==========================================")
    print(alignment_summary)
    print(lambda_frequency)


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# Lambda CV curve from saved tables/
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# ------------------------------------------------------------
# folders
# ------------------------------------------------------------
OUT_DIR = Path("runs/synthetic")
TAB_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# read table
# ------------------------------------------------------------
df = pd.read_csv(
    TAB_DIR / "synthetic_lambda_cv_summary_repeated.csv"
)

# ------------------------------------------------------------
# aggregate
# ------------------------------------------------------------
summary = (
    df.groupby("lambda")
      .agg(
          mean_cv=("cv_srvf_geodesic_mse_mean", "mean"),
          sd_cv=("cv_srvf_geodesic_mse_mean", "std")
      )
      .reset_index()
      .sort_values("lambda")
)

lambdas = summary["lambda"].values
mean_scores = summary["mean_cv"].values
sd_scores = summary["sd_cv"].fillna(0).values

# ------------------------------------------------------------
# best lambda
# ------------------------------------------------------------
best_idx = np.argmin(mean_scores)

best_lambda = lambdas[best_idx]
best_score = mean_scores[best_idx]

print(f"Selected lambda = {best_lambda}")
print(f"Minimum CV error = {best_score:.6f}")

# ------------------------------------------------------------
# figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(
    lambdas,
    mean_scores,
    marker="o",
    linewidth=4,
    markersize=11,
    label="Mean 5-fold CV SRVF geodesic MSE"
)

ax.fill_between(
    lambdas,
    mean_scores - sd_scores,
    mean_scores + sd_scores,
    alpha=0.20,
    label=r"$\pm1$ SD"
)

# ------------------------------------------------------------
# selected lambda
# ------------------------------------------------------------
ax.axvline(
    best_lambda,
    color="red",
    linestyle="--",
    linewidth=2.5,
    alpha=0.9,
    label=rf"Selected $\lambda^\ast={best_lambda:.3g}$"
)

ax.scatter(
    best_lambda,
    best_score,
    color="red",
    s=260,
    marker="*",
    edgecolor="black",
    linewidth=1.2,
    zorder=20
)

ax.annotate(
    rf"$\lambda^\ast={best_lambda:.3g}$",
    xy=(best_lambda, best_score),
    xytext=(15, 15),
    textcoords="offset points",
    fontsize=16,
    color="red",
    weight="bold"
)

# ------------------------------------------------------------
# formatting
# ------------------------------------------------------------
ax.set_xscale("log")

ax.set_xlabel(
    r"$\lambda$",
    fontsize=22
)

ax.set_ylabel(
    "SRVF geodesic validation MSE",
    fontsize=22
)

ax.set_title(
    "5-fold CV lambda selection curve",
    fontsize=26,
    pad=18
)

ax.tick_params(
    axis="both",
    labelsize=18
)

ax.grid(
    True,
    which="both",
    linestyle="--",
    alpha=0.35
)

ax.legend(
    fontsize=15,
    frameon=True,
    loc="upper left"
)

plt.tight_layout()

# ------------------------------------------------------------
# save
# ------------------------------------------------------------
fig.savefig(
    FIG_DIR / "fig_lambda_curve_repeated.pdf",
    bbox_inches="tight"
)

fig.savefig(
    FIG_DIR / "fig_lambda_curve_repeated.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(FIG_DIR / "fig_lambda_curve_repeated.pdf")
print(FIG_DIR / "fig_lambda_curve_repeated.png")